# BERTopic + Image Topic Scoring — Notebook Notes

## Step 0 — Inputs already prepared (webscraping done outside Colab)
Before this notebook starts, we already have **4 CSV files**:

1) **Kremlin English (EN)**  
2) **Kremlin Russian (RU) — includes English translations**  
3) **MID English (EN)**  
4) **MID Russian (RU) — includes English translations**

Each CSV already contains the “scraped + extracted” fields like:
- `id, url, date/year/month/day/time`
- `full_text_word_count`
- `location, latitude, longitude`
- `speakers`, `image_captions`, `page_summary`, `declared_*` columns
- `stored_image_filepaths` (preferred) OR images available in `IMAGE_ROOT/<id>/*`

**RU rule:** we run BERTopic on **`full_text_english` only** (not Russian text), so topic resolution matches EN.

---

## Step 1 — Choose dataset (one run at a time)
Set:
- `DATASET_NAME` ∈ {`kremlin_en`, `kremlin_ru`, `mid_en`, `mid_ru`}
- `CSV_PATH`
- `ID_COL = "id"`
- `TEXT_COL`
  - EN datasets: `TEXT_COL = "full_text"`
  - RU datasets: `TEXT_COL = "full_text_english"`

Images:
- Prefer: `stored_image_filepaths` column (list/delimited string)
- Fallback: folder-per-ID `IMAGE_ROOT/<id>/*`

---

## Step 2 — Fit base BERTopic model once (text only)
For selected dataset:
- Load CSV
- Minimal text cleanup
- Chunk long docs if needed (to avoid embedding crashes)
- Create text embeddings (SentenceTransformer)
- Fit BERTopic → produces a **base model** with `base_k` topics

Artifacts produced (in session):
- `base_model`
- `text_embeddings`
- `topics_base`, `probs_base`

---

## Step 3 — K-sweep (ONLY for EN datasets)
Run K sweep only for:
- Kremlin EN
- MID EN

K sweep logic (clean + fast):
- `K_SWEEP_LIST = list(range(200, 9, -10))`  → 200,190,…,10
- Clamp automatically: only keep `k <= base_k`
- Sweep **descending** because `reduce_topics()` only reduces topics (monotone)
- For each K compute:
  - coherence (c_npmi)
  - diversity
  - compactness
  - separation
  - composite score (custom formula)
- Save table + scree plot

Output:
- `best_k_kremlin_en` (example: 89)
- `best_k_mid_en` (example: 33)

---

## Step 4 — Reuse EN-selected K for RU datasets
No sweep for RU datasets.
Set:
- Kremlin RU uses `K = best_k_kremlin_en`
- MID RU uses `K = best_k_mid_en`

Reason: RU text is translated into English, so we force the same topic resolution.

---

## Step 5 — Fit final BERTopic at chosen K (all 4 datasets)
For each dataset:
- Fit or reduce to selected `K`
- Produce final:
  - `topics_final` (topic id per speech)
  - `probs_final` (topic distribution per speech)

Text outputs:
1) `speech_topk_topics.csv` (wide: one row per speech with top topic + prob)
2) `doc_topic_probs_long.csv` (long: N_docs × K rows)
3) `topics.csv` (topic id + keywords + counts)

---

## Step 6 — Image embeddings + image-topic probability scoring
For each dataset:
- Resolve images per speech:
  - from `stored_image_filepaths` OR `IMAGE_ROOT/<id>/*`
- Embed images using CLIP (cache vectors)
- Build topic prompt vectors (CLIP text) from topic keywords
- Score each image vs each topic → softmax probabilities

Outputs:
1) `image_embeddings.parquet` (manifest: id, image_path, vec_path)
2) `image_topic_probs_long.csv` (long: N_images × K rows)

---

## Step 7 — Build HTML topic browser (topics + images)
For each dataset:
- Use `topics.csv` + `speech_topk_topics.csv`
- Show top speeches per topic
- Show top images per topic (reranked via cached CLIP vectors)
- Generate:
  - `topics_all_in_one.html`

This is used by teammates to:
- name topics
- assign broad groups

---

## Step 8 — Add curated columns (final merge step)
After topic names + group names are finalized, we generate curated columns.

### Text curated columns (per speech)
- `curated_topic_id`
- `curated_text_topic_label`
- `curated_text_topic_group`
- `curated_topic_probabilities` (JSON list length K)

### Image curated columns (per speech)
- `curated_image_topic_ids`
- `curated_image_topic_labels`
- `curated_image_group_names`
- `curated_image_topic_probabilities`

Then produce final CSV per dataset including curated columns.

---

## Step 9 — Final deliverables
At the end we have:
- `kremlin_english.csv`
- `kremlin_russian.csv`
- `mid_english.csv`
- `mid_russian.csv`

Supporting exports (per dataset unless noted):
- `doc_topic_probs_long.csv` (heavy)
- `image_topic_probs_long.csv` (heavy)
- `topics_all_in_one.html`
- EN only: `k_sweep_metrics.csv` + scree plot


## Config: paths + dataset registry + output location

**Purpose**
- Central place to set **all file paths** and **dataset settings** for the 4 datasets:
  - `kremlin_en`, `kremlin_ru`, `mid_en`, `mid_ru`
- Defines the **K-sweep list** (EN only) and where outputs are saved.

**Key rules**
- **EN BERTopic text column:** `full_text`
- **RU BERTopic text column:** `full_text_english` (recommended)
- **K choice:** run K-sweep on EN datasets, then reuse the chosen K for their RU counterparts.

**What you must edit**
- `csv_path` and `image_root` for each dataset.
- If your ID/url column names differ, adjust `id_col_candidates` / `url_col_candidates`.

**Outputs**
- Writes a `config.json` into `OUTPUT_BASE` for reproducibility.


In [ ]:
import os, json
from pathlib import Path

USE_DRIVE = False   # set True if you want outputs saved in Drive
DRIVE_OUT_DIR = "/content/drive/MyDrive/Russian_Speech_BERTopic_Outputs"
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive mount failed:", e)

# Where outputs go
OUTPUT_BASE = DRIVE_OUT_DIR if USE_DRIVE else "/content/outputs"
Path(OUTPUT_BASE).mkdir(parents=True, exist_ok=True)
print("OUTPUT_BASE =", OUTPUT_BASE)

# --------- K sweep list (ONLY for EN datasets) ----------
# Descending because reduce_topics() is monotone (reduces only)
K_SWEEP_LIST = list(range(200, 9, -10))  # 200,190,...,10

# --------- EDIT THESE PATHS (4 CSVs + 4 image roots) ----------
# IMPORTANT:
# - EN datasets: BERTopic reads from `full_text`
# - RU datasets: BERTopic reads from `full_text_english`
DATASETS = {
    "kremlin_en": {
        "csv_path": "/content/drive/MyDrive/.../kremlin_english.csv",   # EDIT
        "text_col": "full_text",
        "image_root": "/content/drive/MyDrive/.../kremlin_english_images",  # EDIT
        "id_col_candidates": ["id", "ID"],
        "url_col_candidates": ["url", "URL"],
        "stored_images_col": "stored_image_filepaths",
        "k_source": None,   # EN decides K by sweep
    },
    "kremlin_ru": {
        "csv_path": "/content/drive/MyDrive/.../kremlin_russian.csv",   # EDIT
        "text_col": "full_text_english",
        "image_root": "/content/drive/MyDrive/.../kremlin_russian_images",  # EDIT
        "id_col_candidates": ["id", "ID"],
        "url_col_candidates": ["url", "URL"],
        "stored_images_col": "stored_image_filepaths",
        "k_source": "kremlin_en",   # RU uses EN chosen K
    },
    "mid_en": {
        "csv_path": "/content/drive/MyDrive/.../mid_english_final.csv", # EDIT
        "text_col": "full_text",
        "image_root": "/content/drive/MyDrive/.../mid_english_scraped_images",  # EDIT
        "id_col_candidates": ["id", "ID"],
        "url_col_candidates": ["url", "URL"],
        "stored_images_col": "stored_image_filepaths",
        "k_source": None,   # EN decides K by sweep
    },
    "mid_ru": {
        "csv_path": "/content/drive/MyDrive/.../mid_russian_final.csv", # EDIT
        "text_col": "full_text_english",
        "image_root": "/content/drive/MyDrive/.../mid_russian_scraped_images",  # EDIT
        "id_col_candidates": ["id", "ID"],
        "url_col_candidates": ["url", "URL"],
        "stored_images_col": "stored_image_filepaths",
        "k_source": "mid_en",   # RU uses EN chosen K
    },
}

# Save config for reproducibility
cfg_path = os.path.join(OUTPUT_BASE, "config.json")
with open(cfg_path, "w", encoding="utf-8") as f:
    json.dump(
        {"USE_DRIVE": USE_DRIVE, "OUTPUT_BASE": OUTPUT_BASE, "K_SWEEP_LIST": K_SWEEP_LIST, "DATASETS": DATASETS},
        f,
        indent=2
    )
print("Saved:", cfg_path)


## Environment setup (Python + required packages)

**Purpose**
- Make sure Colab has a stable set of packages for:
  - BERTopic
  - gensim coherence (c_npmi)
  - UMAP/HDBSCAN
  - sentence-transformers (text + CLIP)
  - pyarrow (parquet)

**Behavior**
- Checks current versions.
- Installs a known-good pinned set **only if needed** (or if you set `FORCE_REINSTALL=True`).
- If it installs anything, it **restarts the runtime automatically** (required so binaries load cleanly).


In [ ]:
import sys, subprocess, importlib
from importlib import metadata

FORCE_REINSTALL = False  # set True only if you want to force pins

PINS = {
    "numpy": "1.26.4",
    "scipy": "1.11.4",
    "pandas": "2.2.2",
    "matplotlib": "3.8.4",
    "scikit-learn": "1.5.2",
    "umap-learn": "0.5.6",
    "hdbscan": "0.8.40",
    "gensim": "4.3.2",
    "bertopic": "0.16.2",
    "sentence-transformers": "3.0.1",
    "tqdm": "4.66.5",
    "pillow": "10.4.0",
    "pyarrow": "17.0.0",
}

def get_ver(pkg: str):
    try:
        return metadata.version(pkg)
    except Exception:
        return None

def pip_install(pkgs):
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "--upgrade"] + pkgs
    subprocess.check_call(cmd)

print("Python:", sys.version)

need = []
for pkg, want in PINS.items():
    have = get_ver(pkg)
    ok = (have == want)
    if FORCE_REINSTALL or (have is None) or (not ok):
        need.append(f"{pkg}=={want}")
    print(f"{pkg:22s} have={have}  want={want}  {'OK' if ok and not FORCE_REINSTALL else 'INSTALL'}")

if need:
    print("\nInstalling pins...")
    pip_install(need)

    # Restart so new wheels load cleanly
    try:
        from google.colab import runtime
        print("Restarting runtime to load fresh binaries...")
        runtime.restart()
    except Exception:
        print("Installed pins. Please restart runtime manually if imports fail later.")
else:
    print("\nAll required packages already match pins. No install needed.")


## Topic label dictionaries (manual curated mapping tables)

**Purpose**
- Store the **topic_id → (topic_label, topic_group)** mappings for each dataset.
- These are NOT written into the CSV now.
- We’ll use them later (final export step) to populate:
  - `curated_topic_id`
  - `curated_text_topic_label`
  - `curated_text_topic_group`
  - and the image curated columns (after image scoring)


In [ ]:
# TOPIC DICTIONARIES (ALL 4 CORPORA)

# ---------- KREMLIN ENGLISH ----------
KREMLIN_EN_TOPIC_DICT = {
    0:  {"label": "Russia's neighbours",                    "group": "IR & Bilateral Relations"},
    1:  {"label": "Domestic coalitions",                    "group": "Executive, Legislature, Judicial"},
    2:  {"label": "Russia-West relations",                  "group": "IR & Bilateral Relations"},
    3:  {"label": "Crimea affairs",                         "group": "Domestic Politics"},
    4:  {"label": "WWII commemoration",                     "group": "Military & Security"},
    5:  {"label": "Putin's European allies",                "group": "IR & Bilateral Relations"},
    6:  {"label": "Intergovernmental cooperation",          "group": "IR & Bilateral Relations"},
    7:  {"label": "Russia-China relations",                 "group": "IR & Bilateral Relations"},
    8:  {"label": "Russian Far East",                       "group": "Domestic Politics"},
    9:  {"label": "Domestic business",                      "group": "Domestic Politics"},
    10: {"label": "Energy sector",                          "group": "Domestic Politics"},
    11: {"label": "Russia-Ukraine relations",               "group": "IR & Bilateral Relations"},
    12: {"label": "Olympics",                               "group": "Sports"},
    13: {"label": "Military and defense",                   "group": "Military & Security"},
    14: {"label": "Domestic economy",                       "group": "Domestic Politics"},
    15: {"label": "Healthcare system",                      "group": "Healthcare"},
    16: {"label": "Non-Western allies",                     "group": "IR & Bilateral Relations"},
    17: {"label": "Religion",                               "group": "Culture & Religion"},
    18: {"label": "National award ceremonies",              "group": "Culture & Religion"},
    19: {"label": "East Asian relations",                   "group": "IR & Bilateral Relations"},
    20: {"label": "Education system",                       "group": "Science & Education"},
    21: {"label": "Russia-ASEAN relations",                 "group": "IR & Bilateral Relations"},
    22: {"label": "Russia-Germany relations",               "group": "IR & Bilateral Relations"},
    23: {"label": "Domestic law enforcement",               "group": "Military & Security"},
    24: {"label": "Russia's navy",                          "group": "Military & Security"},
    25: {"label": "Russia-Eurasian cooperation",            "group": "IR & Bilateral Relations"},
    26: {"label": "Security services",                      "group": "Military & Security"},
    27: {"label": "Russia-European relations",              "group": "IR & Bilateral Relations"},
    28: {"label": "Domestic courts",                        "group": "Executive, Legislature, Judicial"},
    29: {"label": "Russia-Georgia relations",               "group": "IR & Bilateral Relations"},
    30: {"label": "Russia's transportation system",         "group": "Domestic Politics"},
    31: {"label": "Russia-Africa relations",                "group": "IR & Bilateral Relations"},
    32: {"label": "Domestic emergency response",            "group": "Domestic Politics"},
    33: {"label": "Russia-India relations",                 "group": "IR & Bilateral Relations"},
    34: {"label": "Russia-Middle Eastern relations",        "group": "IR & Bilateral Relations"},
    35: {"label": "Domestic financial institutions",        "group": "Domestic Politics"},
    36: {"label": "Russia-Israel-Palestine relations",      "group": "IR & Bilateral Relations"},
    37: {"label": "Russia's culture",                       "group": "Culture & Religion"},
    38: {"label": "Scientific developments",                "group": "Science & Education"},
    39: {"label": "Middle-Eastern partnership",             "group": "IR & Bilateral Relations"},
    40: {"label": "Russia-Iran alliance",                   "group": "IR & Bilateral Relations"},
    41: {"label": "Automotive industry",                    "group": "Domestic Politics"},
    42: {"label": "Space industry",                         "group": "Domestic Politics"},
    43: {"label": "Russia-Latin America relations",         "group": "IR & Bilateral Relations"},
    44: {"label": "Chechnya affairs",                       "group": "Domestic Politics"},
    45: {"label": "Terrorism",                              "group": "Military & Security"},
    46: {"label": "Aviation industry",                      "group": "Military & Security"},
    47: {"label": "Russia-Bulgaria-Greece relations",       "group": "IR & Bilateral Relations"},
    48: {"label": "Russia-Scandinavia relations",           "group": "IR & Bilateral Relations"},
    49: {"label": "Domestic agriculture",                   "group": "Domestic Politics"},
    50: {"label": "Russia-Mongolia relations",              "group": "IR & Bilateral Relations"},
    51: {"label": "World Cup",                              "group": "Sports"},
    52: {"label": "Nuclear industry",                       "group": "Military & Security"},
    53: {"label": "New Year's speeches",                    "group": "Culture & Religion"},
    54: {"label": "Environmental protection",               "group": "Domestic Politics"},
    55: {"label": "Elections",                              "group": "Domestic Politics"},
    56: {"label": "Russia-Spain relations",                 "group": "IR & Bilateral Relations"},
    57: {"label": "Hockey",                                 "group": "Sports"},
    58: {"label": "Russia-Egypt relations",                 "group": "IR & Bilateral Relations"},
    59: {"label": "Domestic volunteerism",                  "group": "Domestic Politics"},
    60: {"label": "Construction industry",                  "group": "Domestic Politics"},
    61: {"label": "Domestic unions",                        "group": "Domestic Politics"},
    62: {"label": "Arctic exploration",                     "group": "Military & Security"},
    63: {"label": "Emergency ministry meetings",            "group": "Domestic Politics"},
    64: {"label": "Media",                                  "group": "Domestic Politics"},
    65: {"label": "Investments",                            "group": "Domestic Politics"},
    66: {"label": "Senior citizens",                        "group": "Domestic Politics"},
    67: {"label": "Financial monitoring",                   "group": "Domestic Politics"},
    68: {"label": "Child welfare policy",                   "group": "Domestic Politics"},
    69: {"label": "Narcotics control",                      "group": "Military & Security"},
    70: {"label": "Family affairs",                         "group": "Domestic Politics"},
    71: {"label": "Russia-Brazil relations",                "group": "IR & Bilateral Relations"},
    72: {"label": "Russia's youth",                         "group": "Domestic Politics"},
    73: {"label": "Border control",                         "group": "Military & Security"},
    74: {"label": "Taxation",                               "group": "Domestic Politics"},
    75: {"label": "Northwestern Europe affairs",            "group": "IR & Bilateral Relations"},
    76: {"label": "Women's recognition",                    "group": "Domestic Politics"},
    77: {"label": "Caspian region",                         "group": "Military & Security"},
    78: {"label": "Russia-Afghanistan relations",           "group": "IR & Bilateral Relations"},
    79: {"label": "Russia-UN relations",                    "group": "IR & Bilateral Relations"},
    80: {"label": "Russia-Cuba relations",                  "group": "IR & Bilateral Relations"},
    81: {"label": "Russia-Indonesia relations",             "group": "IR & Bilateral Relations"},
    82: {"label": "Russian Sports",                         "group": "Sports"},
    83: {"label": "Russia's geographical society",          "group": "Domestic Politics"},
    84: {"label": "Domestic tourism",                       "group": "Domestic Politics"},
    85: {"label": "Students",                               "group": "Science & Education"},
    86: {"label": "Agricultural industry",                  "group": "Domestic Politics"},
    87: {"label": "Russia-Cyprus relations",                "group": "IR & Bilateral Relations"},
    88: {"label": "Customs service",                        "group": "Military & Security"},
}

# ---------- MID ENGLISH (your 32-topic list) ----------
MID_EN_TOPIC_DICT = {
    0:  {"label": "Ukrainian affairs",                     "group": "Post-Soviet Relations"},
    1:  {"label": "Middle Eastern affairs",                "group": "IR & Bilateral Relations"},
    2:  {"label": "Asian affairs",                         "group": "IR & Bilateral Relations"},
    3:  {"label": "Central Asian affairs",                 "group": "Post-Soviet Relations"},
    4:  {"label": "Compatriots affairs",                   "group": "Post-Soviet Relations"},
    5:  {"label": "South Caucasus affairs",                "group": "Post-Soviet Relations"},
    6:  {"label": "Latin American allies",                 "group": "IR & Bilateral Relations"},
    7:  {"label": "African affairs",                       "group": "IR & Bilateral Relations"},
    8:  {"label": "Russian-Islamic states relations",      "group": "IR & Bilateral Relations"},
    9:  {"label": "Iranian nuclear affairs",               "group": "IR & Bilateral Relations"},
    10: {"label": "Russia's economic development",         "group": "Internal affairs"},
    11: {"label": "Arctic affairs",                        "group": "IR & Bilateral Relations"},
    12: {"label": "Eurasian intergovernemntal cooperation", "group": "Post-Soviet Relations"},
    13: {"label": "Developing nations cooperation",        "group": "IR & Bilateral Relations"},
    14: {"label": "Afghanistan relations",                 "group": "IR & Bilateral Relations"},
    15: {"label": "Cyprus-Greece affairs",                 "group": "IR & Bilateral Relations"},
    16: {"label": "Religion",                              "group": "Internal affairs"},
    17: {"label": "Lavrov's interviews",                   "group": "IR & Bilateral Relations"},
    18: {"label": "Korean affairs",                        "group": "IR & Bilateral Relations"},
    19: {"label": "MENA affairs",                          "group": "IR & Bilateral Relations"},
    20: {"label": "WWII commemoration",                    "group": "Internal affairs"},
    21: {"label": "Russia-Germany relations",              "group": "IR & Bilateral Relations"},
    22: {"label": "MGIMO",                                 "group": "Internal affairs"},
    23: {"label": "UNESCO",                                "group": "IR & Bilateral Relations"},
    24: {"label": "Anti-terrorist cooperation",            "group": "IR & Bilateral Relations"},
    25: {"label": "International sports",                  "group": "Sports"},
    26: {"label": "Russia-EU affairs",                     "group": "IR & Bilateral Relations"},
    27: {"label": "Russia-Vietnam relations",              "group": "IR & Bilateral Relations"},
    28: {"label": "Media",                                 "group": "Internal affairs"},
    29: {"label": "Caspian region",                        "group": "Post-Soviet Relations"},
    30: {"label": "Russia-Poland relations",               "group": "IR & Bilateral Relations"},
    31: {"label": "Russia-Lebanon relations",              "group": "IR & Bilateral Relations"},
}

# ---------- KREMLIN RUSSIAN ----------
KREMLIN_RU_TOPIC_DICT = {
    0:  {"label": "Domestic politics",                     "group": "Executive, Legislature, Judicial"},
    1:  {"label": "Russia's neighbours",                   "group": "IR & Bilateral Relations"},
    2:  {"label": "Domestic business",                     "group": "Domestic Politics"},
    3:  {"label": "Russia-West relations",                 "group": "IR & Bilateral Relations"},
    4:  {"label": "Russia-China relations",                "group": "IR & Bilateral Relations"},
    5:  {"label": "Russia-Belarus relations",              "group": "IR & Bilateral Relations"},
    6:  {"label": "Education system",                      "group": "Science & Education"},
    7:  {"label": "Domestic law enforcement",              "group": "Military & Security"},
    8:  {"label": "Russia-Ukraine relations",              "group": "IR & Bilateral Relations"},
    9:  {"label": "Healthcare system",                     "group": "Healthcare"},
    10: {"label": "Russian Sports",                        "group": "Sports"},
    11: {"label": "Russia-Germany relations",              "group": "IR & Bilateral Relations"},
    12: {"label": "Agricultural industry",                 "group": "Domestic Politics"},
    13: {"label": "Putin's European allies",               "group": "IR & Bilateral Relations"},
    14: {"label": "Energy sector",                         "group": "Domestic Politics"},
    15: {"label": "National award ceremonies",             "group": "Culture & Religion"},
    16: {"label": "WWII commemoration",                    "group": "Military & Security"},
    17: {"label": "Eurasian security cooperation",         "group": "Military & Security"},
    18: {"label": "Military and defense",                  "group": "Military & Security"},
    19: {"label": "Crimea affairs",                        "group": "Domestic Politics"},
    20: {"label": "Russian army",                          "group": "Military & Security"},
    21: {"label": "Religion",                              "group": "Culture & Religion"},
    22: {"label": "Russia-Latin America relations",        "group": "IR & Bilateral Relations"},
    23: {"label": "Domestic emergency response",           "group": "Domestic Politics"},
    24: {"label": "Russia's transportation system",        "group": "Domestic Politics"},
    25: {"label": "Technological innovation",              "group": "Science & Education"},
    26: {"label": "Armed Forces",                          "group": "Military & Security"},
    27: {"label": "Middle-Eastern partnership",            "group": "IR & Bilateral Relations"},
    28: {"label": "Security services",                     "group": "Military & Security"},
    29: {"label": "Nuclear industry",                      "group": "Military & Security"},
    30: {"label": "New Year's speeches",                   "group": "Culture & Religion"},
    31: {"label": "Russia-European relations",             "group": "IR & Bilateral Relations"},
    32: {"label": "Scientific developments",               "group": "Science & Education"},
    33: {"label": "Non-Western allies",                    "group": "IR & Bilateral Relations"},
    34: {"label": "Russia's culture",                      "group": "Culture & Religion"},
    35: {"label": "Russia-Africa relations",               "group": "IR & Bilateral Relations"},
    36: {"label": "Russia's navy",                         "group": "Military & Security"},
    37: {"label": "Civil society and human rights",        "group": "Domestic Politics"},
    38: {"label": "Russia-India relations",                "group": "IR & Bilateral Relations"},
    39: {"label": "Russia-Georgia relations",              "group": "IR & Bilateral Relations"},
    40: {"label": "Russia-ASEAN relations",                "group": "IR & Bilateral Relations"},
    41: {"label": "Russia-Israel-Palestine relations",     "group": "IR & Bilateral Relations"},
    42: {"label": "Space industry",                        "group": "Domestic Politics"},
    43: {"label": "Domestic courts",                       "group": "Executive, Legislature, Judicial"},
    44: {"label": "World Cup",                             "group": "Sports"},
    45: {"label": "Construction industry",                 "group": "Domestic Politics"},
    46: {"label": "Environmental protection",              "group": "Domestic Politics"},
    47: {"label": "Demographic policy",                    "group": "Domestic Politics"},
    48: {"label": "Russia-Scandinavia relations",          "group": "IR & Bilateral Relations"},
    49: {"label": "Interethnic relations",                 "group": "Domestic Politics"},
    50: {"label": "Russia-Mongolia relations",             "group": "IR & Bilateral Relations"},
    51: {"label": "Technology",                            "group": "Science & Education"},
    52: {"label": "Russia-Middle Eastern relations",       "group": "IR & Bilateral Relations"},
    53: {"label": "Russia-Eurasian cooperation",           "group": "IR & Bilateral Relations"},
    54: {"label": "Aviation industry",                     "group": "Military & Security"},
    55: {"label": "Russia-Greece-Cyprus relations",        "group": "IR & Bilateral Relations"},
    56: {"label": "Domestic unions",                       "group": "Domestic Politics"},
    57: {"label": "Russia-Afghanistan relations",          "group": "IR & Bilateral Relations"},
    58: {"label": "Domestic volunteerism",                 "group": "Domestic Politics"},
    59: {"label": "Global economy",                        "group": "IR & Bilateral Relations"},
    60: {"label": "Anti-corruption policy",                "group": "Domestic Politics"},
    61: {"label": "Russian Far East",                      "group": "Domestic Politics"},
    62: {"label": "Elections",                             "group": "Domestic Politics"},
    63: {"label": "Senior citizens",                       "group": "Domestic Politics"},
    64: {"label": "Olympics",                              "group": "Sports"},
    65: {"label": "Russia-Bulgaria relations",             "group": "IR & Bilateral Relations"},
    66: {"label": "Narcotics control",                     "group": "Military & Security"},
    67: {"label": "Taxation",                              "group": "Domestic Politics"},
    68: {"label": "Border control",                        "group": "Military & Security"},
    69: {"label": "Financial monitoring",                  "group": "Domestic Politics"},
    70: {"label": "Russia's geographical society",         "group": "Domestic Politics"},
    71: {"label": "Russia-Brazil relations",               "group": "IR & Bilateral Relations"},
    72: {"label": "Media",                                 "group": "Domestic Politics"},
    73: {"label": "Investments",                           "group": "Domestic Politics"},
    74: {"label": "Russia-Indonesia relations",            "group": "IR & Bilateral Relations"},
    75: {"label": "Russia-Cuba relations",                 "group": "IR & Bilateral Relations"},
    76: {"label": "Arctic exploration",                    "group": "Military & Security"},
    77: {"label": "Caspian region",                        "group": "Military & Security"},
    78: {"label": "Customs service",                       "group": "Military & Security"},
    79: {"label": "Russia-Poland relations",               "group": "IR & Bilateral Relations"},
    80: {"label": "Nazi atrocities",                       "group": "Military & Security"},
    81: {"label": "Students",                              "group": "Science & Education"},
    82: {"label": "Social policy",                         "group": "Domestic Politics"},
    83: {"label": "Martial arts",                          "group": "Sports"},
    84: {"label": "Family affairs",                        "group": "Domestic Politics"},
    85: {"label": "Women's recognition",                   "group": "Domestic Politics"},
    86: {"label": "Land policy",                           "group": "Domestic Politics"},
    87: {"label": "Russia-UN relations",                   "group": "IR & Bilateral Relations"},
    88: {"label": "Russia-Canada relations",               "group": "IR & Bilateral Relations"},
}

# ---------- MID RUSSIAN ----------
MID_RU_TOPIC_DICT = {
    0:  {"label": "Ukrainian affairs",              "group": "Post-Soviet Relations"},
    1:  {"label": "Middle Eastern affairs",         "group": "IR & Bilateral Relations"},
    2:  {"label": "Asian affairs",                  "group": "IR & Bilateral Relations"},
    3:  {"label": "Compatriots affairs",            "group": "Post-Soviet Relations"},
    4:  {"label": "African affairs",                "group": "IR & Bilateral Relations"},
    5:  {"label": "Central Asian affairs",          "group": "Post-Soviet Relations"},
    6:  {"label": "Latin American allies",          "group": "IR & Bilateral Relations"},
    7:  {"label": "Iranian nuclear affairs",        "group": "IR & Bilateral Relations"},
    8:  {"label": "European security",              "group": "IR & Bilateral Relations"},
    9:  {"label": "Religion",                       "group": "Internal affairs"},
    10: {"label": "Black Sea cooperation",          "group": "IR & Bilateral Relations"},
    11: {"label": "South Caucasus affairs",         "group": "Post-Soviet Relations"},
    12: {"label": "Arctic affairs",                 "group": "IR & Bilateral Relations"},
    13: {"label": "Eurasian intergovernemntal cooperation", "group": "Post-Soviet Relations"},
    14: {"label": "Developing nations cooperation", "group": "IR & Bilateral Relations"},
    15: {"label": "Cyprus-Greece affairs",          "group": "IR & Bilateral Relations"},
    16: {"label": "Afghanistan relations",          "group": "IR & Bilateral Relations"},
    17: {"label": "WWII commemoration",             "group": "Internal affairs"},
    18: {"label": "Korean affairs",                 "group": "IR & Bilateral Relations"},
    19: {"label": "Nazism",                         "group": "IR & Bilateral Relations"},
    20: {"label": "Russia-Germany relations",       "group": "IR & Bilateral Relations"},
    21: {"label": "Domestic sport",                 "group": "Internal affairs"},
    22: {"label": "ASEAN relations",                "group": "IR & Bilateral Relations"},
    23: {"label": "Regional policy",                "group": "Internal affairs"},
    24: {"label": "MGIMO",                          "group": "Internal affairs"},
    25: {"label": "UNESCO",                         "group": "IR & Bilateral Relations"},
    26: {"label": "Lavrov's interviews",            "group": "IR & Bilateral Relations"},
    27: {"label": "Russia-Italy relations",         "group": "IR & Bilateral Relations"},
    28: {"label": "Caspian region",                 "group": "Post-Soviet Relations"},
    29: {"label": "Anti-terrorist cooperation",     "group": "IR & Bilateral Relations"},
    30: {"label": "Anti-narcotics trafficking",     "group": "IR & Bilateral Relations"},
    31: {"label": "Humanitarian cooperation",       "group": "IR & Bilateral Relations"},
}

# ---------- Convenience map by corpus key ----------
TOPIC_DICT_BY_CORPUS = {
    "kremlin_en": KREMLIN_EN_TOPIC_DICT,
    "kremlin_ru": KREMLIN_RU_TOPIC_DICT,
    "mid_en":     MID_EN_TOPIC_DICT,
    "mid_ru":     MID_RU_TOPIC_DICT,
}

print("Loaded topic dictionaries:",
      f"KREMLIN_EN={len(KREMLIN_EN_TOPIC_DICT)} | "
      f"KREMLIN_RU={len(KREMLIN_RU_TOPIC_DICT)} | "
      f"MID_EN={len(MID_EN_TOPIC_DICT)} | "
      f"MID_RU={len(MID_RU_TOPIC_DICT)}")

## Run configuration (paths + dataset switch)

**Purpose**
- Choose which dataset to run (`mid_en`, `mid_ru`, `kremlin_en`, `kremlin_ru`)
- Set CSV path and image root
- Decide which text column BERTopic reads:
  - EN: `full_text`
  - RU (translated): `full_text_english`

In [ ]:
import os
from datetime import datetime

# --- Choose dataset key ---
DATASET_KEY = "mid_en"       # one of: "mid_en", "mid_ru", "kremlin_en", "kremlin_ru"

# --- Input CSVs (EDIT THESE) ---
CSV_PATHS = {
    "mid_en":     "/content/drive/MyDrive/Russian Speech Dataset Project/New Files/mid_english.csv",
    "mid_ru":     "/content/drive/MyDrive/Russian Speech Dataset Project/New Files/mid_russian.csv",
    "kremlin_en": "/content/drive/MyDrive/Russian Speech Dataset Project/New Files/kremlin_english.csv",
    "kremlin_ru": "/content/drive/MyDrive/Russian Speech Dataset Project/New Files/kremlin_russian.csv",
}

# --- Image roots (EDIT THESE) ---
IMAGE_ROOTS = {
    "mid_en":     "/content/drive/MyDrive/Russian Speech Dataset Project/New Files/mid_english_images",
    "mid_ru":     "/content/drive/MyDrive/Russian Speech Dataset Project/New Files/mid_russian_images",
    "kremlin_en": "/content/drive/MyDrive/Russian Speech Dataset Project/New Files/kremlin_english_images",
    "kremlin_ru": "/content/drive/MyDrive/Russian Speech Dataset Project/New Files/kremlin_russian_images",
}

# EN datasets: "full_text"
# RU datasets (translated): "full_text_english"
TEXT_COL_BY_DATASET = {
    "mid_en": "full_text",
    "kremlin_en": "full_text",
    "mid_ru": "full_text_english",
    "kremlin_ru": "full_text_english",
}

# --- K selection rules ---
# Sweep only for EN datasets; RU uses the chosen K from its paired EN.
K_SWEEP_MIN = 10
K_SWEEP_MAX = 200
K_SWEEP_STEP = 10
K_SWEEP_LIST_DESC = list(range(K_SWEEP_MAX, K_SWEEP_MIN - 1, -K_SWEEP_STEP))  # 200,190,...,10

# Final chosen Ks (these were set after inspecting the sweep plots/metrics)
CHOSEN_K = {
    "mid_en": 33,
    "kremlin_en": 89,
    "mid_ru": 33,
    "kremlin_ru": 89,
}

# --- Output folder ---
RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_ROOT = os.path.abspath(f"./outputs/{DATASET_KEY}_{RUN_STAMP}")
os.makedirs(OUT_ROOT, exist_ok=True)

CSV_PATH = CSV_PATHS[DATASET_KEY]
IMAGE_ROOT = IMAGE_ROOTS[DATASET_KEY]
TEXT_COL = TEXT_COL_BY_DATASET[DATASET_KEY]
TOPIC_DICT = TOPIC_DICT_BY_DATASET[DATASET_KEY]

print("DATASET_KEY:", DATASET_KEY)
print("CSV_PATH   :", CSV_PATH)
print("IMAGE_ROOT :", IMAGE_ROOT)
print("TEXT_COL   :", TEXT_COL)
print("OUT_ROOT   :", OUT_ROOT)
print("Topic dict topics:", len(TOPIC_DICT))
print("Sweep list (desc) sample:", K_SWEEP_LIST_DESC[:5], "...", K_SWEEP_LIST_DESC[-3:])
print("Chosen K for this dataset:", CHOSEN_K.get(DATASET_KEY))

## Load dataset + validate columns + resolve image paths

**Purpose**
- Load the selected dataset CSV (`CSV_PATH`)
- Normalize column names (`id`, `url`, `stored_image_filepaths`)
- Build `docs` from `TEXT_COL`
- Resolve images per row using:
  1) `stored_image_filepaths` (supports `||` or `|` separators)
  2) fallback to folder-per-ID: `IMAGE_ROOT/<id>/*`
- Print sanity stats (rows, empty docs, images found)


In [ ]:
import os, glob, re
import numpy as np
import pandas as pd

IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

def _normp(p: str) -> str:
    return os.path.normpath(str(p)).replace("\\", "/")

def _coalesce_cols(df: pd.DataFrame, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def load_csv_utf8(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, encoding="utf-8", dtype=str, low_memory=False)
    df.columns = [c.strip() for c in df.columns]
    return df

def normalize_schema(df: pd.DataFrame) -> pd.DataFrame:
    id_col = _coalesce_cols(df, ["id", "ID", "Id"])
    if not id_col:
        raise ValueError("CSV must contain an 'id' or 'ID' column.")
    df = df.rename(columns={id_col: "id"})
    df["id"] = df["id"].astype(str).str.strip()

    url_col = _coalesce_cols(df, ["url", "URL", "link"])
    if url_col and url_col != "url":
        df = df.rename(columns={url_col: "url"})
    if "url" not in df.columns:
        df["url"] = ""
    df["url"] = df["url"].astype(str).fillna("").str.strip()

    # ---- stored_image_filepaths ----
    # sometimes people name it image_filenames / image_paths etc
    img_col = _coalesce_cols(df, ["stored_image_filepaths", "image_filenames", "image_paths"])
    if img_col and img_col != "stored_image_filepaths":
        df = df.rename(columns={img_col: "stored_image_filepaths"})
    if "stored_image_filepaths" not in df.columns:
        df["stored_image_filepaths"] = ""

    # ---- keep text col present ----
    if TEXT_COL not in df.columns:
        raise ValueError(f"TEXT_COL='{TEXT_COL}' not found in CSV columns. Found: {list(df.columns)[:30]} ...")

    df[TEXT_COL] = df[TEXT_COL].astype(str).fillna("").replace({"nan": ""})

    return df

import os, json, ast

IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

def split_image_cell_list_only(cell, image_root: str, strict: bool = True) -> list:

    if cell is None:
        return []

    # If already a Python list/tuple
    if isinstance(cell, (list, tuple)):
        parts = list(cell)
    else:
        s = str(cell).strip()
        if not s or s.lower() in ("nan", "none", "null"):
            return []

        if not (s.startswith("[") and s.endswith("]")):
            if strict:
                raise ValueError(f"Expected list-style string like ['a','b'] but got: {s[:120]}")
            return []

        try:
            parts = json.loads(s)
        except Exception:
            try:
                parts = ast.literal_eval(s)
            except Exception as e:
                if strict:
                    raise ValueError(f"Could not parse list-style image cell: {s[:120]}") from e
                return []

    if not isinstance(parts, (list, tuple)):
        if strict:
            raise ValueError(f"Parsed value is not a list/tuple. Got type={type(parts)}")
        return []

    out = []
    for p in parts:
        if p is None:
            continue
        p = str(p).strip().strip('"').strip("'")
        if not p:
            continue

        cand = p if os.path.isabs(p) else os.path.join(image_root, p)
        cand = os.path.abspath(cand)

        ext = os.path.splitext(cand.lower())[1]
        if ext in IMG_EXTS:
            out.append(cand)

    return out

def list_images_folder_per_id(speech_id: str) -> list:
    folder = os.path.join(IMAGE_ROOT, str(speech_id).strip())
    if not os.path.isdir(folder):
        return []
    paths = []
    for p in sorted(glob.glob(os.path.join(folder, "*"))):
        if os.path.splitext(p.lower())[1] in IMG_EXTS and os.path.exists(p):
            paths.append(os.path.abspath(p))
    return paths

def resolve_images_for_row(row: pd.Series) -> list:
    paths = split_image_cell(row.get("stored_image_filepaths", ""))
    exists = [p for p in paths if os.path.exists(p)]
    if exists:
        return exists
    return list_images_folder_per_id(row["id"])

df = load_csv_utf8(CSV_PATH)
df = normalize_schema(df)

docs = df[TEXT_COL].astype(str).fillna("").tolist()

url_map = dict(zip(df["id"].tolist(), df["url"].tolist()))

all_paths_per_row = []
for _, r in df.iterrows():
    all_paths_per_row.append(resolve_images_for_row(r))
df["_all_image_paths"] = all_paths_per_row

all_images = sorted({p for paths in all_paths_per_row for p in paths if os.path.exists(p)})

n = len(df)
nonempty_docs = sum(1 for t in docs if isinstance(t, str) and t.strip())
empty_docs = n - nonempty_docs
rows_with_images = sum(1 for paths in all_paths_per_row if len(paths) > 0)

print("Rows:", n)
print("Docs non-empty:", nonempty_docs, "| empty:", empty_docs)
print("Rows with >=1 image:", rows_with_images)
print("Unique existing images found:", len(all_images))

print("\nColumns:", list(df.columns))
print("\nSample row:")
print(df[["id", "url", TEXT_COL, "stored_image_filepaths", "_all_image_paths"]].head(1).to_dict(orient="records")[0])


## Build **text embeddings** (chunk long docs safely)

**Purpose**
- Create one embedding vector per row (speech) for BERTopic.
- Handles long `full_text` by **token-chunking** (avoids the “629 > 512” warning).
- Saves `text_embeddings.npz` (X = float32 embeddings) + `chunk_stats.csv` in Colab session (`OUT_DIR`).


In [ ]:
import os, math, re
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from sentence_transformers import SentenceTransformer

os.makedirs(OUT_DIR, exist_ok=True)

TEXT_EMB_PATH   = os.path.join(OUT_DIR, "text_embeddings.npz")
CHUNK_STATS_CSV = os.path.join(OUT_DIR, "chunk_stats.csv")

# If you rerun often, set True to reuse saved embeddings
REUSE_EMBEDDINGS = True

assert "TEXT_EMBED_MODEL" in globals(), "Define TEXT_EMBED_MODEL in config cell first."
assert "docs" in globals(), "Run Cell 5 first to build docs list."

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
print("Text embedding model:", TEXT_EMBED_MODEL)

def clean_text_basic(x: str) -> str:
    x = str(x or "").replace("\xa0", " ")
    x = re.sub(r"\s+", " ", x).strip()
    return x

def chunk_by_tokens(text: str, tokenizer, max_tokens: int):
    """
    Chunk by tokens using the model tokenizer so no chunk exceeds max_tokens.
    We reserve room for special tokens implicitly by just using max_tokens.
    """
    text = clean_text_basic(text)
    if not text:
        return []

    # encode without truncation
    ids = tokenizer.encode(text, add_special_tokens=False)
    if not ids:
        return []

    chunks = []
    for i in range(0, len(ids), max_tokens):
        part_ids = ids[i:i + max_tokens]
        chunk = tokenizer.decode(part_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True)
        chunk = clean_text_basic(chunk)
        if chunk:
            chunks.append(chunk)
    return chunks

def l2_normalize(X: np.ndarray):
    n = np.linalg.norm(X, axis=1, keepdims=True)
    n[n == 0.0] = 1.0
    return X / n

if REUSE_EMBEDDINGS and os.path.exists(TEXT_EMB_PATH):
    emb = np.load(TEXT_EMB_PATH)["X"].astype(np.float32)
    print("Loaded existing embeddings:", emb.shape, "from", TEXT_EMB_PATH)
else:
    model = SentenceTransformer(TEXT_EMBED_MODEL, device=DEVICE)

    tok = getattr(model, "tokenizer", None)
    if tok is None:
        raise RuntimeError("SentenceTransformer tokenizer not found; cannot token-chunk safely.")

    max_len = getattr(model, "max_seq_length", None)
    if not isinstance(max_len, int) or max_len <= 0:
        max_len = 512

    CHUNK_MAX_TOKENS = max(128, min(480, max_len - 16))

    # 1) Chunk all docs
    chunked = []
    chunk_counts = []
    for t in tqdm(docs, desc="Chunking docs"):
        parts = chunk_by_tokens(t, tok, CHUNK_MAX_TOKENS)
        chunked.append(parts)
        chunk_counts.append(len(parts))

    # 2) Flatten chunks for encoding
    flat_texts = [c for parts in chunked for c in parts]
    print("Total docs:", len(docs))
    print("Total chunks:", len(flat_texts))
    print("Avg chunks/doc:", float(np.mean(chunk_counts)) if chunk_counts else 0.0)

    # 3) Encode all chunks
    BATCH = 256 if DEVICE == "cuda" else 64
    flat_vecs = []

    for i in tqdm(range(0, len(flat_texts), BATCH), desc="Encoding chunks"):
        batch = flat_texts[i:i + BATCH]
        vec = model.encode(
            batch,
            batch_size=min(BATCH, len(batch)),
            show_progress_bar=False,
            convert_to_numpy=True
        ).astype(np.float32)
        flat_vecs.append(vec)

    flat_vecs = np.vstack(flat_vecs) if flat_vecs else np.zeros((0, 384), dtype=np.float32)

    # 4) Aggregate chunks -> doc embedding (mean of L2-normalized chunk embeddings)
    idx = 0
    doc_vecs = []
    for parts in chunked:
        if not parts:
            # empty doc -> zeros
            doc_vecs.append(np.zeros((flat_vecs.shape[1],), dtype=np.float32))
            continue
        m = len(parts)
        V = flat_vecs[idx:idx + m]
        idx += m
        V = l2_normalize(V)
        v = V.mean(axis=0).astype(np.float32)
        doc_vecs.append(v)

    emb = np.vstack(doc_vecs).astype(np.float32)
    np.savez_compressed(TEXT_EMB_PATH, X=emb)
    print("Saved embeddings:", emb.shape, "to", TEXT_EMB_PATH)

    # Save chunk stats
    stats = pd.DataFrame({
        "id": df["id"].astype(str).tolist(),
        "n_chunks": chunk_counts,
        "text_len_chars": [len(clean_text_basic(t)) for t in docs],
    })
    stats.to_csv(CHUNK_STATS_CSV, index=False)
    print("Saved chunk stats:", CHUNK_STATS_CSV)

assert emb.shape[0] == len(docs), "Embeddings rows must match docs rows."
print("Embeddings ready:", emb.shape)


## Fit **Base BERTopic (Text-Only)** + save base artifacts

**Purpose**
- Fit BERTopic **once** on `(docs, embeddings)` to get the **base** model (max topics).
- Save the model + base outputs (topic info, doc→topic + probabilities).
- This base model will be used later for **K-sweep (EN only)** via `reduce_topics()` (monotone reductions, no refits).


In [ ]:
import os, json, random
import numpy as np
import pandas as pd

from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from sklearn.feature_extraction.text import CountVectorizer

import umap
import hdbscan

# ---- Repro ----
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
except Exception:
    pass

# ---- Paths ----
BASE_MODEL_DIR   = os.path.join(OUT_DIR, "topic_model_text_only")
TOPIC_INFO_CSV   = os.path.join(OUT_DIR, "base_topic_info.csv")
DOC_TOPICS_CSV   = os.path.join(OUT_DIR, "base_doc_topics.csv")          # per-doc top topic
DOC_PROBS_NPZ    = os.path.join(OUT_DIR, "base_doc_topic_probs.npz")     # full probs matrix (big but compact)

# If rerun often, reuse
REUSE_BASE_MODEL = True

assert "df" in globals() and "docs" in globals(), "Run Cell 5 first."
assert "emb" in globals(), "Run Cell 6 first to build text embeddings."

ids = df["id"].astype(str).tolist()

def save_base_outputs(model: BERTopic, topics, probs):
    # Topic info
    info = model.get_topic_info()
    info.to_csv(TOPIC_INFO_CSV, index=False)

    # Doc -> top topic + top prob
    top_prob = None
    if probs is not None and isinstance(probs, np.ndarray) and probs.ndim == 2 and probs.shape[0] == len(ids):
        # probs[i, topic_index] aligns with model.get_topics() order; BERTopic keeps topic ids as columns in probs
        # easiest: take max prob row-wise (ignores -1; still ok for ranking)
        top_prob = probs.max(axis=1).astype(np.float32)

    out = pd.DataFrame({
        "id": ids,
        "top_topic": topics,
        "top_prob": top_prob if top_prob is not None else np.nan,
    })
    out.to_csv(DOC_TOPICS_CSV, index=False)

    # Save full probs (heavy). Keep compressed npz.
    if probs is not None and isinstance(probs, np.ndarray):
        np.savez_compressed(DOC_PROBS_NPZ, probs=probs.astype(np.float32))

    print("Saved:", TOPIC_INFO_CSV)
    print("Saved:", DOC_TOPICS_CSV)
    if os.path.exists(DOC_PROBS_NPZ):
        print("Saved:", DOC_PROBS_NPZ)

# ---- Load or fit ----
if REUSE_BASE_MODEL and os.path.isdir(BASE_MODEL_DIR):
    print("Loading existing base model:", BASE_MODEL_DIR)
    base_model = BERTopic.load(BASE_MODEL_DIR)
    topics, probs = base_model.transform(docs, embeddings=emb)
    save_base_outputs(base_model, topics, probs)
else:
    print("Fitting base BERTopic model...")

    # UMAP + HDBSCAN (stable defaults; adjust later only if needed)
    umap_model = umap.UMAP(
        n_neighbors=15,
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=SEED,
    )

    hdbscan_model = hdbscan.HDBSCAN(
        min_cluster_size=15,
        min_samples=None,
        metric="euclidean",
        cluster_selection_method="eom",
        prediction_data=True,
    )

    # Vectorizer (EN stopwords because RU datasets use translated English text too)
    vectorizer_model = CountVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        min_df=5
    )

    ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)

    base_model = BERTopic(
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        ctfidf_model=ctfidf_model,
        calculate_probabilities=True,
        verbose=True
    )

    topics, probs = base_model.fit_transform(docs, embeddings=emb)

    # Save model
    os.makedirs(BASE_MODEL_DIR, exist_ok=True)
    base_model.save(BASE_MODEL_DIR)
    print("Saved base model to:", BASE_MODEL_DIR)

    save_base_outputs(base_model, topics, probs)

topic_ids = [t for t in base_model.get_topics().keys() if t != -1]
print("Base topics (excluding -1):", len(topic_ids))
print("Example topic info head:")
display(pd.read_csv(TOPIC_INFO_CSV).head(10))


## K-sweep (EN only): reduce_topics() from 200→10 and compute composite score + plot

**Run this cell only when** `DATASET_TAG` is an **English** dataset (Kremlin_EN or MID_EN).
For RU datasets, we will **skip** K-sweep and directly reduce to the chosen K (same K as its EN partner).

**What it does**
- Loads the **base** model from Cell 7.
- Gets initial `(topics0, probs0)` once.
- Runs a **descending** K list: `200, 190, …, 10` (auto-clamped to `<= base_k`).
- Uses **monotone reductions**
- Saves:
  - `k_sweep_metrics.csv`
  - `k_scree_plot.png`
  - prints best-K by composite score


In [ ]:
import os, re, copy, inspect, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import minmax_scale
from sklearn.metrics.pairwise import cosine_similarity

from gensim.utils import simple_preprocess
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
from gensim.parsing.preprocessing import STOPWORDS

from bertopic import BERTopic

assert "base_model" in globals(), "Run Cell 7 first."
assert "docs" in globals() and "emb" in globals(), "Run Cells 5–6 first."
assert "DATASET_TAG" in globals(), "Run Cell 3 first (config)."

if "RU" in DATASET_TAG.upper():
    print(f"Skipping K-sweep because DATASET_TAG={DATASET_TAG} is RU. Use EN's chosen K.")
else:
    SWEEP_DIR  = os.path.join(OUT_DIR, "k_sweep")
    os.makedirs(SWEEP_DIR, exist_ok=True)
    METRICS_CSV = os.path.join(SWEEP_DIR, "k_sweep_metrics.csv")
    PLOT_PNG    = os.path.join(SWEEP_DIR, "k_scree_plot.png")

    # Repro
    SEED = 42
    os.environ["PYTHONHASHSEED"] = str(SEED)
    random.seed(SEED)
    np.random.seed(SEED)
    try:
        import torch
        torch.manual_seed(SEED)
        torch.cuda.manual_seed_all(SEED)
    except Exception:
        pass

    # Tokens once (for coherence)
    def clean_text(t: str) -> str:
        t = "" if t is None else str(t)
        t = t.replace("\xa0", " ")
        t = re.sub(r"\s+", " ", t).strip()
        return t

    tokens = [simple_preprocess(clean_text(t), deacc=True, min_len=2, max_len=30) for t in docs]
    dictionary = Dictionary(tokens)

    TOP_WORDS = 10
    MIN_DOCS_PER_TOPIC = 2

    def topic_words_fallback(model, tid, member_idx, topn=TOP_WORDS):
        pairs = model.get_topic(tid) or []
        words = [w for w,_ in pairs[:topn] if isinstance(w, str)]
        if len(words) >= max(3, min(topn, 5)):
            return words[:topn]

        from collections import Counter
        ctr = Counter()
        for i in member_idx:
            ctr.update([w for w in tokens[i] if w not in STOPWORDS])
        fallback = [w for w,_ in ctr.most_common(300) if len(w) >= 2 and not w.isdigit()]
        seen = set(words)
        words = words + [w for w in fallback if w not in seen]
        return words[:topn]

    def coherence_npmi_safe(wordlists):
        try:
            if not wordlists:
                return 0.0
            cm = CoherenceModel(
                topics=wordlists, texts=tokens, dictionary=dictionary,
                coherence="c_npmi", processes=1
            )
            v = float(cm.get_coherence())
            return 0.0 if not np.isfinite(v) else v
        except Exception:
            return 0.0

    def metrics_at(model, assignments):
        all_tids = [t for t in model.get_topics().keys() if t != -1]
        members = {tid: [] for tid in all_tids}
        for i, t in enumerate(assignments):
            if t in members:
                members[t].append(i)

        tids = [tid for tid in all_tids if len(members[tid]) >= MIN_DOCS_PER_TOPIC]
        if not tids:
            return dict(coherence=0.0, diversity=0.0, compactness=0.0, separation=0.0)

        wlists = []
        for tid in tids:
            ws = topic_words_fallback(model, tid, members[tid], topn=TOP_WORDS)
            if ws:
                wlists.append(ws)

        coherence = coherence_npmi_safe(wlists)

        flat = [w for ws in wlists for w in ws]
        diversity = (len(set(flat)) / len(flat)) if flat else 0.0

        cents = {}
        for tid in tids:
            idx = members[tid]
            if idx:
                cents[tid] = emb[idx].mean(axis=0)

        sims = []
        for tid in tids:
            idx = members[tid]
            if not idx or tid not in cents:
                continue
            sims.extend(
                cosine_similarity(emb[idx], cents[tid].reshape(1, -1)).ravel().tolist()
            )
        compactness = float(np.mean(sims)) if sims else 0.0

        if len(cents) > 1:
            C = np.vstack([cents[tid] for tid in cents])
            S = cosine_similarity(C)
            D = 1.0 - S
            separation = float(np.mean(D[np.triu_indices(D.shape[0], 1)]))
        else:
            separation = 0.0

        return dict(coherence=coherence, diversity=diversity, compactness=compactness, separation=separation)

    def reduce_topics_compat(m, docs, topics, probs, k):
        fn = m.reduce_topics
        sig = inspect.signature(fn)
        params = list(sig.parameters.keys())
        try:
            if {"docs","topics","probabilities","nr_topics"}.issubset(params):
                out = fn(docs=docs, topics=topics, probabilities=probs, nr_topics=int(k))
            elif {"docs","nr_topics"}.issubset(params):
                out = fn(docs=docs, nr_topics=int(k))
            else:
                out = fn(docs, topics, probs, int(k))
        except TypeError:
            try:
                out = fn(docs, topics, probs, int(k))
            except Exception:
                out = fn(docs, nr_topics=int(k))

        if isinstance(out, tuple):
            if len(out) == 3:
                mk, tk, pk = out
            elif len(out) == 2:
                mk, tk = out
                pk = None
            else:
                mk = out[0]
                tk, pk = mk.transform(docs, embeddings=emb)
        else:
            mk = out
            tk, pk = mk.transform(docs, embeddings=emb)

        return mk, tk, pk

    # ---- K list: 200..10 (desc), clamp to base_k ----
    base_k = len([t for t in base_model.get_topics().keys() if t != -1])
    K_SWEEP_LIST = list(range(200, 9, -10))  # 200,190,...,10
    K_LIST_DESC  = [k for k in K_SWEEP_LIST if k <= base_k]
    if not K_LIST_DESC:
        K_LIST_DESC = [base_k]
    print(f"base_k={base_k} | K sweep (desc): {K_LIST_DESC}")

    # Start from base assignments once
    seed_model = copy.deepcopy(base_model)
    topics0, probs0 = seed_model.transform(docs, embeddings=emb)

    rows = []
    curr_m, curr_t, curr_p = seed_model, topics0, probs0
    curr_k = base_k

    for k in K_LIST_DESC:
        if k == curr_k:
            mk, tk, pk = curr_m, curr_t, curr_p
        else:
            mk, tk, pk = reduce_topics_compat(curr_m, docs, curr_t, curr_p, k)

        metr = metrics_at(mk, tk)
        rows.append({"K": int(k), **metr})
        curr_m, curr_t, curr_p = mk, tk, pk
        curr_k = k
        print(f"K={k} → c_npmi={metr['coherence']:.4f}  div={metr['diversity']:.4f}  comp={metr['compactness']:.4f}  sep={metr['separation']:.4f}")

    res = pd.DataFrame(rows).sort_values("K")  # ascending for plotting

    # Composite score weights
    W = {"coherence":0.40, "diversity":0.20, "compactness":0.25, "separation":0.15}
    def norm(x):
        x = np.asarray(x, float)
        return np.zeros_like(x) if np.allclose(x.max(), x.min()) else minmax_scale(x)

    res["coherence_norm"]   = norm(res["coherence"])
    res["diversity_norm"]   = norm(res["diversity"])
    res["compactness_norm"] = norm(res["compactness"])
    res["separation_norm"]  = norm(res["separation"])
    res["score"] = (
        W["coherence"]   * res["coherence_norm"] +
        W["diversity"]   * res["diversity_norm"] +
        W["compactness"] * res["compactness_norm"] +
        W["separation"]  * res["separation_norm"]
    )

    res.to_csv(METRICS_CSV, index=False)
    print("Saved metrics:", METRICS_CSV)

    # Plot
    plt.figure(figsize=(10.5, 6.0))
    Ks = res["K"].values
    plt.plot(Ks, res["coherence_norm"],   marker="o", label="Coherence (c_npmi, norm)")
    plt.plot(Ks, res["diversity_norm"],   marker="o", label="Diversity (norm)")
    plt.plot(Ks, res["compactness_norm"], marker="o", label="Compactness (norm)")
    plt.plot(Ks, res["separation_norm"],  marker="o", label="Separation (norm)")
    plt.plot(Ks, res["score"],            marker="o", linewidth=3, label="Composite score", alpha=0.85)

    bi = int(res["score"].idxmax())
    bk = int(res.loc[bi, "K"])
    by = float(res.loc[bi, "score"])
    plt.scatter([bk], [by], s=160, marker="*", zorder=5, label=f"Chosen K={bk}")

    plt.xlabel("Number of topics (K)")
    plt.ylabel("Normalized value / score")
    plt.title(f"K sweep: 200→10 step 10 (monotone reduce_topics) | {DATASET_TAG}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(PLOT_PNG, dpi=160)
    plt.close()
    print("Saved scree plot:", PLOT_PNG)

    print("\nTop rows by score:")
    display(res.sort_values("score", ascending=False).head(10)[["K","coherence","diversity","compactness","separation","score"]])

    CHOSEN_K = bk
    print(f"\n CHOSEN_K for {DATASET_TAG} = {CHOSEN_K}")


## Pick final K for this run (EN from K-sweep, RU from its paired EN)

**Goal:** set `FINAL_K` that we will use to build the final reduced model + exports.

Rules:
- If `DATASET_TAG` is **EN**: use `CHOSEN_K`.
- If `DATASET_TAG` is **RU**: set `FINAL_K` equal to the **paired EN** K:
  - Kremlin_RU → Kremlin_EN K (89)
  - MID_RU     → MID_EN K (33)

We have hardcoded these two numbers as Kremlin(89) and MID(33), as we got these values.


In [ ]:
assert "DATASET_TAG" in globals(), "Run Cell 3 first (config)."

KREMLIN_EN_FINAL_K = 89
MID_EN_FINAL_K     = 33

tag = DATASET_TAG.upper()

if "KREMLIN_EN" in tag:
    FINAL_K = globals().get("CHOSEN_K", KREMLIN_EN_FINAL_K)
elif "MID_EN" in tag:
    FINAL_K = globals().get("CHOSEN_K", MID_EN_FINAL_K)
elif "KREMLIN_RU" in tag:
    FINAL_K = KREMLIN_EN_FINAL_K
elif "MID_RU" in tag:
    FINAL_K = MID_EN_FINAL_K
else:
    raise ValueError(f"Unknown DATASET_TAG={DATASET_TAG}")

# manual override
# FINAL_K = 89

print(f"FINAL_K for {DATASET_TAG} = {FINAL_K}")


## Build final reduced BERTopic model (no refits) + save core outputs

**Goal:** Reduce the already-fit base model down to `FINAL_K` using `reduce_topics()` (monotone reduce, no re-embedding), then export:
- `topics.csv` (topic_id, keywords, count, optional topic_name placeholder)
- `speech_topk_topics.csv` (per speech: top_topic, top_prob)
- `doc_topic_probs_long.csv` (heavy: every doc × every topic probability)
- `topic_model_reduced/` (saved reduced BERTopic model)

**Important:**
- Uses the same `reduce_topics_compat()` you already used (handles BERTopic version differences).
- Uses `emb` from `text_embeddings.npz` so transform is consistent.


In [ ]:
import os, json
import numpy as np
import pandas as pd

assert "docs" in globals() and "emb" in globals() and "base_model" in globals(), "Run Cells 4–6 first."
assert "FINAL_K" in globals(), "Run Cell 9 first."
assert "reduce_topics_compat" in globals(), "Run Cell 6 first."

OUT_FINAL = os.path.join(OUTPUT_DIR, f"FINAL_K_{int(FINAL_K)}")
os.makedirs(OUT_FINAL, exist_ok=True)

# 1) Get base assignments once
topics0, probs0 = base_model.transform(docs, embeddings=emb)

# 2) Reduce to FINAL_K (no refit)
base_k = len([t for t in base_model.get_topics().keys() if t != -1])
if FINAL_K > base_k:
    print(f"FINAL_K={FINAL_K} > base_k={base_k}. Clamping to base_k.")
    FINAL_K = base_k

reduced_model, topics_k, probs_k = reduce_topics_compat(base_model, docs, topics0, probs0, int(FINAL_K))

# 3) Save reduced model (folder)
MODEL_OUT_DIR = os.path.join(OUT_FINAL, "topic_model_reduced")
reduced_model.save(MODEL_OUT_DIR)
print("Saved reduced model:", MODEL_OUT_DIR)

# 4) Build topics.csv
# keywords from model.get_topic(tid)
topic_rows = []
topic_info = reduced_model.get_topic_info()  # includes Count per topic
topic_info = topic_info[topic_info["Topic"] != -1].copy()

for _, r in topic_info.iterrows():
    tid = int(r["Topic"])
    cnt = int(r["Count"])
    pairs = reduced_model.get_topic(tid) or []
    keywords = ", ".join([w for w, _ in pairs[:10] if isinstance(w, str)])
    topic_rows.append({
        "topic_id": tid,
        "count": cnt,
        "keywords": keywords,
        "topic_name": ""
    })

topics_df = pd.DataFrame(topic_rows).sort_values("topic_id")
TOPICS_CSV = os.path.join(OUT_FINAL, "topics.csv")
topics_df.to_csv(TOPICS_CSV, index=False)
print("topics.csv:", TOPICS_CSV)

# 5) speech_topk_topics.csv (per doc: top topic + prob)
# probs_k can be None depending on reduce_topics behavior; if None, recompute
if probs_k is None:
    topics_k, probs_k = reduced_model.transform(docs, embeddings=emb)

# Normalize shapes safely
probs_k = np.asarray(probs_k)
if probs_k.ndim != 2:
    raise ValueError(f"Unexpected probs_k shape: {probs_k.shape}")

# Row IDs
id_col = "id" if "id" in df.columns else ("ID" if "ID" in df.columns else None)
if id_col is None:
    raise ValueError("CSV must contain id/ID.")
ids = df[id_col].astype(str).tolist()

top_topic = probs_k.argmax(axis=1)
top_prob  = probs_k.max(axis=1)

speech_top = pd.DataFrame({
    "id": ids,
    "top_topic": top_topic.astype(int),
    "top_prob": top_prob.astype(float),
})
SPEECH_TOP_CSV = os.path.join(OUT_FINAL, "speech_topk_topics.csv")
speech_top.to_csv(SPEECH_TOP_CSV, index=False)
print("speech_topk_topics.csv:", SPEECH_TOP_CSV)

# 6) doc_topic_probs_long.csv
# every doc × every topic prob
topic_ids = topics_df["topic_id"].astype(int).tolist()

# probs_k columns are topic indices in BERTopic order (0..K-1).
# We keep numeric topic IDs from the reduced model; assume they align with probs columns by sorted topic_ids.
# To be robust: map column order using reduced_model.get_topic_info() order
ordered_topics = topic_info.sort_values("Topic")["Topic"].astype(int).tolist()
if len(ordered_topics) != probs_k.shape[1]:
    # Fallback: assume probs columns correspond to 0..K-1
    ordered_topics = list(range(probs_k.shape[1]))

rows = []
for i, sid in enumerate(ids):
    for j, tid in enumerate(ordered_topics):
        rows.append({
            "id": sid,
            "topic": int(tid),
            "prob": float(probs_k[i, j]),
        })

doc_topic_long = pd.DataFrame(rows)
DOC_LONG_CSV = os.path.join(OUT_FINAL, "doc_topic_probs_long.csv")
doc_topic_long.to_csv(DOC_LONG_CSV, index=False)
print("doc_topic_probs_long.csv:", DOC_LONG_CSV)
print("Rows:", len(doc_topic_long))

# 7) Save a small run metadata json
meta = {
    "dataset_tag": DATASET_TAG,
    "final_k": int(FINAL_K),
    "base_k": int(base_k),
    "model_dir": MODEL_OUT_DIR,
    "topics_csv": TOPICS_CSV,
    "speech_top_csv": SPEECH_TOP_CSV,
    "doc_topic_long_csv": DOC_LONG_CSV,
}
META_JSON = os.path.join(OUT_FINAL, "run_meta.json")
with open(META_JSON, "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)
print("run_meta.json:", META_JSON)


## Build Topics HTML (text topics + image gallery)

**Goal:** Create a single `topics_all_in_one.html` that shows:
- Topic header (topic_id, count, keywords)
- Top speeches (by assigned probability)
- Top images per topic (re-ranked with CLIP prompt vs cached image vectors)
- This does **NOT** re-embed images (uses cached `vec_path`).
- It only embeds the topic text prompts once with CLIP.

**Inputs expected (already created earlier):**
- `FINAL_K_* / topics.csv`
- `FINAL_K_* / speech_topk_topics.csv`
- Image manifest parquet from the image-vector pipeline (ID, image_path, vec_path)
- The same dataset CSV used for BERTopic (for URLs)




In [ ]:
import os, base64
from io import BytesIO
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image, ImageOps
import torch
from sentence_transformers import SentenceTransformer
from datetime import datetime

assert "OUT_FINAL" in globals(), "Run Cell 10 first (OUT_FINAL not found)."

CSV_PATH_FOR_URLS = CSV_PATH  # same input CSV you used for BERTopic text
MANIFEST_PARQUET  = MANIFEST_PARQUET if "MANIFEST_PARQUET" in globals() else None
# MANIFEST_PARQUET = "/content/drive/MyDrive/.../image_embeddings_<tag>.parquet"

CLIP_MODEL = "clip-ViT-B-32"

# Controls
IMAGES_PER_TOPIC       = 10
MAX_IMAGES_PER_SPEECH  = 8
TEMPERATURE            = 0.07   # only used if you later want softmax; here we use cosine ranking
THUMB_W                = 200

topics_csv = os.path.join(OUT_FINAL, "topics.csv")
speech_csv = os.path.join(OUT_FINAL, "speech_topk_topics.csv")
if not os.path.exists(topics_csv): raise FileNotFoundError(topics_csv)
if not os.path.exists(speech_csv): raise FileNotFoundError(speech_csv)

topics_df = pd.read_csv(topics_csv)
speech_df = pd.read_csv(speech_csv)
df_docs   = pd.read_csv(CSV_PATH_FOR_URLS, encoding="utf-8", dtype=str, low_memory=False)

# Image manifest
if MANIFEST_PARQUET and os.path.exists(MANIFEST_PARQUET):
    img_manifest = pd.read_parquet(MANIFEST_PARQUET)
else:
    img_manifest = pd.DataFrame(columns=["ID","image_path","vec_path"])

# Normalize ID columns
def norm_id_col(df, cand=("id","ID")):
    for c in cand:
        if c in df.columns:
            return c
    return None

docs_id_col = norm_id_col(df_docs)
if docs_id_col is None:
    raise ValueError("Input CSV must have id/ID column for HTML linking.")

df_docs[docs_id_col] = df_docs[docs_id_col].astype(str)
speech_df["id"] = speech_df["id"].astype(str)

if "ID" in img_manifest.columns:
    img_manifest["ID"] = img_manifest["ID"].astype(str)
elif "id" in img_manifest.columns:
    img_manifest["ID"] = img_manifest["id"].astype(str)
else:
    # keep empty / already normalized
    pass

def load_resize_image(path: str, size=(384, 384)):
    try:
        im = Image.open(path).convert("RGB")
        im = ImageOps.exif_transpose(im)
        im = im.resize(size, getattr(Image, "LANCZOS", Image.BICUBIC))
        return im
    except Exception:
        return None

def encode_png_base64(im: Image.Image):
    buf = BytesIO()
    im.save(buf, format="PNG", optimize=False)
    return base64.b64encode(buf.getvalue()).decode("utf-8")

def l2(x):
    n = np.linalg.norm(x, axis=1, keepdims=True)
    n[n == 0.0] = 1.0
    return x / n

def topic_prompt(keywords: str) -> str:
    return f"news photo of {keywords}"

def make_tile(path, sid, link, thumb_w=THUMB_W):
    im = load_resize_image(path)
    if im is None:
        return ""
    b64 = encode_png_base64(im)
    link_html = f' — <a href="{link}" target="_blank">open</a>' if (link and isinstance(link,str) and link.strip()) else ""
    return (
        f'<figure style="width:{thumb_w}px;margin:6px;">'
        f'  <img loading="lazy" src="data:image/png;base64,{b64}" '
        f'       style="width:{thumb_w}px;border:1px solid #ddd;border-radius:6px;">'
        f'  <figcaption style="font:12px/1.35 system-ui;color:#444;margin-top:4px;">ID <b>{sid}</b>{link_html}</figcaption>'
        f'</figure>'
    )

# Build topic members (top_prob ranking)
id_to_row = {str(v): i for i, v in enumerate(df_docs[docs_id_col].astype(str).tolist())}

topic_members = {}  # tid -> list of (prob, row_idx)
for _, r in speech_df.iterrows():
    tid  = int(r["top_topic"])
    sid  = str(r["id"])
    prob = float(r["top_prob"]) if pd.notnull(r["top_prob"]) else 0.0
    if sid in id_to_row:
        topic_members.setdefault(tid, []).append((prob, id_to_row[sid]))

for t in topic_members:
    topic_members[t].sort(key=lambda x: -x[0])

# Prepare CLIP text encoder for prompts
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
clip_text = SentenceTransformer(CLIP_MODEL, device=DEVICE)

# Embed one prompt per topic from keywords
prompts, tids = [], []
for _, r in topics_df.iterrows():
    tid = int(r["topic_id"])
    kw  = str(r.get("keywords",""))
    prompts.append(topic_prompt(kw))
    tids.append(tid)

prompt_vecs = {}
if prompts:
    P = clip_text.encode(prompts, show_progress_bar=False, convert_to_numpy=True).astype(np.float32)
    P = l2(P)
    prompt_vecs = {tids[i]: P[i] for i in range(len(tids))}

# Rerank images per topic using cached vectors
def rerank_images_for_topic(tid: int, topk: int = IMAGES_PER_TOPIC, cap: int = 300):
    if tid not in prompt_vecs or img_manifest.empty:
        return []
    pvec = prompt_vecs[tid]
    cand = []

    members = topic_members.get(tid, [])
    for prob, row_idx in members:
        sid = str(df_docs.loc[row_idx, docs_id_col])

        row_imgs = img_manifest[img_manifest["ID"].astype(str) == sid]
        if row_imgs.empty:
            continue

        link = None
        if "url" in df_docs.columns and isinstance(df_docs.loc[row_idx, "url"], str) and df_docs.loc[row_idx, "url"].strip():
            link = df_docs.loc[row_idx, "url"]

        cnt = 0
        for _, rr in row_imgs.iterrows():
            vecp = rr.get("vec_path", None)
            imgp = rr.get("image_path", None)
            if not isinstance(vecp, str) or not os.path.exists(vecp):
                continue
            if not isinstance(imgp, str) or not os.path.exists(imgp):
                continue

            v = np.load(vecp).astype(np.float32)
            v = v / (np.linalg.norm(v) + 1e-12)
            score = float(np.dot(pvec, v))
            cand.append((score, sid, imgp, link))

            cnt += 1
            if cnt >= MAX_IMAGES_PER_SPEECH:
                break
        if len(cand) >= cap:
            break

    cand.sort(key=lambda x: -x[0])
    return cand[:topk]

# Build HTML
sections = []
for _, r in topics_df.iterrows():
    tid   = int(r["topic_id"])
    name  = str(r.get("topic_name","")).strip()
    count = int(r.get("count",0))
    keys  = str(r.get("keywords",""))

    ranked = rerank_images_for_topic(tid, IMAGES_PER_TOPIC, cap=300)
    tiles  = [make_tile(p, sid, link) for (score, sid, p, link) in ranked]
    gallery = ''.join(tiles) if tiles else '<div style="color:#777;">No images available</div>'

    # Top speeches table (first 6)
    rows_html = []
    for pr, idx in topic_members.get(tid, [])[:6]:
        sid = str(df_docs.loc[idx, docs_id_col])
        if "url" in df_docs.columns and isinstance(df_docs.loc[idx, "url"], str) and df_docs.loc[idx, "url"].strip():
            url_html = f'<a href="{df_docs.loc[idx, "url"]}" target="_blank">{sid}</a>'
        else:
            url_html = sid
        rows_html.append(
            f"<tr><td style='padding:6px 10px;border-bottom:1px solid #eee'>{url_html}</td>"
            f"<td style='padding:6px 10px;border-bottom:1px solid #eee;text-align:right'>{pr:.3f}</td></tr>"
        )
    top_table = (
        "<table style='border-collapse:collapse;font:13px system-ui;'>"
        "<thead><tr><th style='text-align:left;padding:6px 10px;border-bottom:1px solid #ddd'>Speech ID</th>"
        "<th style='text-align:right;padding:6px 10px;border-bottom:1px solid #ddd'>Assigned prob</th></tr></thead>"
        f"<tbody>{''.join(rows_html)}</tbody></table>"
    )

    title = f"Topic {tid}" + (f" — {name}" if name else "")
    section_html = f"""
    <section id="topic-{tid}" style="margin:24px 0 36px 0;padding-top:24px;border-top:2px solid #f2f2f2;">
      <h2 style="margin:0 0 6px 0;">{title}</h2>
      <div style="color:#666;margin:0 0 12px 0;">Count: <b>{count}</b></div>
      <div style="color:#444;margin:0 0 14px 0;"><b>Keywords:</b> {keys}</div>
      <div style="display:flex;flex-wrap:wrap;gap:6px;">{gallery}</div>
      <details style="margin-top:14px;">
        <summary style="cursor:pointer;">Top speeches (first 6)</summary>
        <div style="margin-top:10px;">{top_table}</div>
      </details>
      <div style="margin-top:14px;"><a href="#top">Back to top</a></div>
    </section>
    """
    sections.append(section_html)

toc_items = [
    f'<li><a href="#topic-{int(r["topic_id"])}">Topic {int(r["topic_id"])}</a></li>'
    for _, r in topics_df.iterrows()
]
toc_html  = f"<ol>{''.join(toc_items)}</ol>"

stamp = datetime.now().strftime("%Y-%m-%d %H:%M")

html = f"""<!doctype html><meta charset="utf-8">
<title>Topics (FINAL K={int(FINAL_K)})</title>
<div id="top" style="font-family:system-ui,-apple-system,Segoe UI,Roboto,Arial;padding:18px;max-width:1220px;margin:0 auto;">
  <h1 style="margin:0 0 8px 0;">Topics Overview — FINAL K={int(FINAL_K)}</h1>
  <p style="color:#666;margin-top:0;">Reduced model. Images re-ranked with CLIP using cached vectors (no re-embedding).</p>
  <div style="background:#fafafa;border:1px solid #eee;border-radius:8px;padding:12px;margin:12px 0;">
    <h3 style="margin:0 0 8px 0;">Contents</h3>
    {toc_html}
  </div>
  {''.join(sections)}
  <hr style="margin:24px 0;">
  <div style="color:#888;font:12px system-ui;">
    Generated: {stamp} • CLIP: {CLIP_MODEL} • Images/topic: {IMAGES_PER_TOPIC}
  </div>
</div>"""

out_html = os.path.join(OUT_FINAL, "topics_all_in_one.html")
with open(out_html, "w", encoding="utf-8") as f:
    f.write(html)

print("Saved HTML →", out_html)


## Export Long-Format Topic Probability Tables (Text + Images)

**Goal:** Create the “long” tables you described:
- **Text:** `doc_topic_probs_long.csv` → for every speech × every topic (N_speeches × K rows)
- **Images:** `image_topic_probs_long.csv` → for every image × every topic (N_images × K rows)

**Important:**
- Uses the **FINAL reduced model** (K = `FINAL_K`) and the **same embeddings** (no refits).
- For **RU datasets**, this uses whatever `TEXT_COL` is set to (recommended: `full_text_english`).
- Image long-table uses **cached image vectors** from `MANIFEST_PARQUET` (`vec_path`) and topic prompt vectors from keywords.

You can disable either export if it’s too heavy.


In [ ]:
import os, math, json
import numpy as np
import pandas as pd
from tqdm import tqdm

from bertopic import BERTopic

assert "OUT_FINAL" in globals(), "Run Cell 10 first (OUT_FINAL not found)."
assert "CSV_PATH" in globals(), "CSV_PATH not found."
assert "TEXT_COL" in globals(), "TEXT_COL not found."
assert "EMB_PATH" in globals(), "EMB_PATH not found."
assert "FINAL_K" in globals(), "FINAL_K not found."

EXPORT_TEXT_LONG   = True   # doc_topic_probs_long.csv (heavy)
EXPORT_IMAGE_LONG  = True   # image_topic_probs_long.csv (very heavy)

# Chunk sizes
DOC_CHUNK = 500       # speeches per chunk
IMG_CHUNK = 200       # images per chunk

# ---------- INPUTS ----------
MODEL_FINAL_PATH = os.path.join(OUT_FINAL, "topic_model_text_only")
if not os.path.exists(MODEL_FINAL_PATH):
    raise FileNotFoundError(f"Final model not found: {MODEL_FINAL_PATH}")

df = pd.read_csv(CSV_PATH, encoding="utf-8", dtype=str, low_memory=False)
if TEXT_COL not in df.columns:
    raise ValueError(f"TEXT_COL='{TEXT_COL}' not found in CSV columns.")

# ID column normalize
ID_COL = "id" if "id" in df.columns else ("ID" if "ID" in df.columns else None)
if ID_COL is None:
    raise ValueError("Input CSV must contain 'id' or 'ID' column.")
df[ID_COL] = df[ID_COL].astype(str)

docs = df[TEXT_COL].fillna("").astype(str).tolist()

emb_npz = np.load(EMB_PATH)

emb = emb_npz["X"].astype(np.float32)

if len(docs) != emb.shape[0]:
    raise ValueError(f"Docs count ({len(docs)}) != embedding rows ({emb.shape[0]}).")

# ---------- LOAD FINAL MODEL ----------
topic_model = BERTopic.load(MODEL_FINAL_PATH)

# Transform once
topics, probs = topic_model.transform(docs, embeddings=emb)

# probs columns order is not guaranteed by label; we build a robust col→topic_id mapping:
# for each topic_id, find which prob-column is highest on average for docs assigned to that topic.
def build_col_to_topic_mapping(topics_arr, probs_arr):
    unique_tids = sorted({int(t) for t in set(topics_arr) if int(t) != -1})
    n_cols = probs_arr.shape[1]
    col_map = {}
    used_cols = set()

    for tid in unique_tids:
        idx = np.where(topics_arr == tid)[0]
        if len(idx) == 0:
            continue
        meanp = probs_arr[idx].mean(axis=0)
        order = np.argsort(-meanp)  # best column first
        for c in order:
            if int(c) not in used_cols:
                col_map[int(c)] = int(tid)
                used_cols.add(int(c))
                break

    col_to_tid = [col_map.get(c, None) for c in range(n_cols)]

    # for a sample of docs, argmax column should map to assigned topic
    ok, checked = 0, 0
    for i in range(min(2000, len(topics_arr))):
        if int(topics_arr[i]) == -1:
            continue
        c = int(np.argmax(probs_arr[i]))
        mapped = col_to_tid[c]
        if mapped == int(topics_arr[i]):
            ok += 1
        checked += 1
        if checked >= 200:
            break
    print(f"[Mapping check] matched {ok}/{checked} sample docs.")
    return col_to_tid

col_to_tid = build_col_to_topic_mapping(np.array(topics, dtype=int), probs)

# Final topic id list in column order
valid_cols = [i for i, tid in enumerate(col_to_tid) if tid is not None]
topic_ids_in_col_order = [col_to_tid[i] for i in valid_cols]

# TEXT LONG EXPORT
if EXPORT_TEXT_LONG:
    out_text_long = os.path.join(OUT_FINAL, "doc_topic_probs_long.csv")
    if os.path.exists(out_text_long):
        os.remove(out_text_long)

    print("Writing:", out_text_long)
    header_written = False

    ids = df[ID_COL].tolist()
    n_docs = len(ids)
    K = len(valid_cols)

    for start in tqdm(range(0, n_docs, DOC_CHUNK), desc="Text long export"):
        end = min(start + DOC_CHUNK, n_docs)

        chunk_ids = ids[start:end]
        chunk_probs = probs[start:end, :][:, valid_cols]  # (chunk, K)

        # build long arrays
        rep_ids   = np.repeat(np.array(chunk_ids, dtype=object), K)
        rep_tids  = np.tile(np.array(topic_ids_in_col_order, dtype=int), (end-start))
        rep_probs = chunk_probs.reshape(-1)

        out_df = pd.DataFrame({
            "id": rep_ids,
            "topic_id": rep_tids,
            "topic_prob": rep_probs
        })

        out_df.to_csv(out_text_long, index=False, mode="a", header=(not header_written))
        header_written = True

    print("Saved text long table:", out_text_long)
else:
    print("Skipping text long export (EXPORT_TEXT_LONG=False).")

# IMAGE LONG EXPORT
if EXPORT_IMAGE_LONG:
    # We need topics.csv (keywords) + MANIFEST_PARQUET (vec_path)
    topics_csv = os.path.join(OUT_FINAL, "topics.csv")
    if not os.path.exists(topics_csv):
        raise FileNotFoundError(f"Missing topics.csv: {topics_csv}")

    if "MANIFEST_PARQUET" not in globals() or (not isinstance(MANIFEST_PARQUET, str)) or (not os.path.exists(MANIFEST_PARQUET)):
        raise FileNotFoundError(
            "MANIFEST_PARQUET missing/not found. Set MANIFEST_PARQUET to your image embeddings parquet."
        )

    topics_df = pd.read_csv(topics_csv)
    if "topic_id" not in topics_df.columns:
        raise ValueError("topics.csv must contain topic_id.")
    if "keywords" not in topics_df.columns:
        raise ValueError("topics.csv must contain keywords.")

    img_manifest = pd.read_parquet(MANIFEST_PARQUET)
    # Normalize ID column
    if "ID" in img_manifest.columns:
        img_manifest["ID"] = img_manifest["ID"].astype(str)
    elif "id" in img_manifest.columns:
        img_manifest["ID"] = img_manifest["id"].astype(str)
    else:
        raise ValueError("Image manifest must contain ID or id column.")

    if "vec_path" not in img_manifest.columns or "image_path" not in img_manifest.columns:
        raise ValueError("Image manifest must have vec_path and image_path columns.")

    # Build prompt vectors in the SAME column order as our text topics mapping
    # We'll only use topics that exist in col_to_tid mapping (topic IDs)
    want_topic_ids = sorted(set(topic_ids_in_col_order))
    topics_df = topics_df[topics_df["topic_id"].astype(int).isin(want_topic_ids)].copy()
    topics_df["topic_id"] = topics_df["topic_id"].astype(int)

    # Reorder topics_df to match numeric topic_id order, then we build P in that order.
    topics_df = topics_df.sort_values("topic_id")
    topic_list_for_images = topics_df["topic_id"].tolist()

    # Encode prompts with CLIP text model
    import torch
    from sentence_transformers import SentenceTransformer

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    clip_text = SentenceTransformer("clip-ViT-B-32", device=DEVICE)

    prompts = [f"news photo of {str(k)}" for k in topics_df["keywords"].fillna("").astype(str).tolist()]
    P = clip_text.encode(prompts, show_progress_bar=True, convert_to_numpy=True).astype(np.float32)
    P = P / (np.linalg.norm(P, axis=1, keepdims=True) + 1e-12)  # (K, D)

    def softmax_stable(x, temp=0.07):
        x = x / max(temp, 1e-6)
        x = x - np.max(x)
        ex = np.exp(x)
        return ex / (np.sum(ex) + 1e-12)

    out_img_long = os.path.join(OUT_FINAL, "image_topic_probs_long.csv")
    if os.path.exists(out_img_long):
        os.remove(out_img_long)

    print("Writing:", out_img_long)
    header_written = False

    # We stream images in chunks
    rows = img_manifest[["ID","image_path","vec_path"]].copy()
    rows = rows.dropna()
    rows["ID"] = rows["ID"].astype(str)

    total = len(rows)
    for start in tqdm(range(0, total, IMG_CHUNK), desc="Image long export"):
        end = min(start + IMG_CHUNK, total)
        sub = rows.iloc[start:end]

        out_records = []
        for _, rr in sub.iterrows():
            sid = rr["ID"]
            imgp = rr["image_path"]
            vecp = rr["vec_path"]

            if not isinstance(vecp, str) or not os.path.exists(vecp):
                continue

            v = np.load(vecp).astype(np.float32)
            v = v / (np.linalg.norm(v) + 1e-12)  # (D,)

            sims = (P @ v).astype(np.float32)  # (K,)
            probs_img = softmax_stable(sims, temp=0.07)

            # write K rows per image
            for k_i, tid in enumerate(topic_list_for_images):
                out_records.append({
                    "id": sid,
                    "image_path": imgp,
                    "topic_id": int(tid),
                    "topic_prob": float(probs_img[k_i])
                })

        chunk_df = pd.DataFrame(out_records)
        chunk_df.to_csv(out_img_long, index=False, mode="a", header=(not header_written))
        header_written = True

    print("Saved image long table:", out_img_long)
else:
    print("Skipping image long export (EXPORT_IMAGE_LONG=False).")


## Create Final “Curated” CSV (Add Curated Text + Curated Image Columns)

**Goal:** Starting from your input CSV, add these final columns and write **one final CSV**:
- `curated_topic_id`
- `curated_text_topic_label`
- `curated_text_topic_group`
- `curated_topic_probabilities`  *(stores TOP-N topic probs per speech; change TOPN_STORE to store more)*

And for images (aligned to `stored_image_filepaths` order):
- `curated_image_topic_ids`
- `curated_image_topic_labels`
- `curated_image_group_names`
- `curated_image_topic_probabilities`

**Assumptions:**
- You already ran the **final model** (Cell 10) and you have `OUT_FINAL`.
- `TOPIC_DICT` is set for the current dataset (Kremlin EN / MID EN / Kremlin RU / MID RU).
- Your CSV has `stored_image_filepaths`.
- `MANIFEST_PARQUET` exists and contains `image_path` + `vec_path` (from the image embedding step).

This cell does **not** redo webscraping/location/speaker extraction — it just adds curated topic outputs.


In [ ]:
import os, re, json
import numpy as np
import pandas as pd
from tqdm import tqdm

# ---- required globals from earlier cells ----
assert "OUT_FINAL" in globals(), "Run Cell 10 first (OUT_FINAL not found)."
assert "CSV_PATH" in globals(), "CSV_PATH not found."
assert "TEXT_COL" in globals(), "TEXT_COL not found."
assert "EMB_PATH" in globals(), "EMB_PATH not found."
assert "FINAL_K" in globals(), "FINAL_K not found."
assert "TOPIC_DICT" in globals(), "TOPIC_DICT not found. Set TOPIC_DICT for this dataset."

MODEL_FINAL_PATH = os.path.join(OUT_FINAL, "topic_model_text_only")
if not os.path.exists(MODEL_FINAL_PATH):
    raise FileNotFoundError(f"Final model not found: {MODEL_FINAL_PATH}")

# ---- knobs ----
TOPN_STORE = 10     # set to FINAL_K if you truly want all topics stored per speech
PROB_FMT = "{:.6f}" # probability formatting
IMAGE_TEMP = 0.07   # softmax temperature for image-topic probs

FINAL_CURATED_CSV = os.path.join(OUT_FINAL, "final_curated.csv")

df = pd.read_csv(CSV_PATH, encoding="utf-8", dtype=str, low_memory=False)

ID_COL = "id" if "id" in df.columns else ("ID" if "ID" in df.columns else None)
if ID_COL is None:
    raise ValueError("Input CSV must contain 'id' or 'ID' column.")
df[ID_COL] = df[ID_COL].astype(str)

docs = df[TEXT_COL].fillna("").astype(str).tolist()

emb_npz = np.load(EMB_PATH)
emb = emb_npz["X"].astype(np.float32)
if len(docs) != emb.shape[0]:
    raise ValueError(f"Docs count ({len(docs)}) != embedding rows ({emb.shape[0]}).")

from bertopic import BERTopic
topic_model = BERTopic.load(MODEL_FINAL_PATH)

# Reuse if already computed in memory
reuse_ok = ("topics" in globals()) and ("probs" in globals()) and (len(globals()["topics"]) == len(docs))
if reuse_ok:
    topics_arr = np.array(globals()["topics"], dtype=int)
    probs_arr  = np.array(globals()["probs"], dtype=np.float32)
else:
    topics_arr, probs_arr = topic_model.transform(docs, embeddings=emb)
    topics_arr = np.array(topics_arr, dtype=int)
    probs_arr  = np.array(probs_arr, dtype=np.float32)

# robust mapping: prob-column -> topic_id
def build_col_to_topic_mapping(topics_arr, probs_arr):
    unique_tids = sorted({int(t) for t in set(topics_arr) if int(t) != -1})
    n_cols = probs_arr.shape[1]
    col_map = {}
    used_cols = set()

    for tid in unique_tids:
        idx = np.where(topics_arr == tid)[0]
        if len(idx) == 0:
            continue
        meanp = probs_arr[idx].mean(axis=0)
        order = np.argsort(-meanp)
        for c in order:
            if int(c) not in used_cols:
                col_map[int(c)] = int(tid)
                used_cols.add(int(c))
                break

    col_to_tid = [col_map.get(c, None) for c in range(n_cols)]

    # quick sanity check
    ok, checked = 0, 0
    for i in range(min(3000, len(topics_arr))):
        if int(topics_arr[i]) == -1:
            continue
        c = int(np.argmax(probs_arr[i]))
        if col_to_tid[c] == int(topics_arr[i]):
            ok += 1
        checked += 1
        if checked >= 300:
            break
    print(f"[Mapping check] matched {ok}/{checked} sample docs.")
    return col_to_tid

col_to_tid = build_col_to_topic_mapping(topics_arr, probs_arr)
valid_cols = [i for i, tid in enumerate(col_to_tid) if tid is not None]
topic_ids_in_col_order = [col_to_tid[i] for i in valid_cols]

def label_for_tid(tid: int) -> str:
    d = TOPIC_DICT.get(int(tid), None)
    return d.get("label", "") if isinstance(d, dict) else ""

def group_for_tid(tid: int) -> str:
    d = TOPIC_DICT.get(int(tid), None)
    return d.get("group", "") if isinstance(d, dict) else ""

cur_topic_id = []
cur_lbl = []
cur_grp = []
cur_probs = []

K = len(valid_cols)
TOPN = min(int(TOPN_STORE), K)

for i in tqdm(range(len(df)), desc="Curated text cols"):
    tid = int(topics_arr[i])
    cur_topic_id.append("" if tid == -1 else str(tid))
    cur_lbl.append("" if tid == -1 else label_for_tid(tid))
    cur_grp.append("" if tid == -1 else group_for_tid(tid))

    if tid == -1:
        cur_probs.append("")
        continue

    rowp = probs_arr[i, :][valid_cols]  # (K,)
    # top-N
    idx = np.argsort(-rowp)[:TOPN]
    parts = [f"{int(topic_ids_in_col_order[j])}:{PROB_FMT.format(float(rowp[j]))}" for j in idx]
    cur_probs.append("||".join(parts))

df["curated_topic_id"] = cur_topic_id
df["curated_text_topic_label"] = cur_lbl
df["curated_text_topic_group"] = cur_grp
df["curated_topic_probabilities"] = cur_probs

# If you don't have images for this dataset, you can skip safely.
HAS_IMAGES = ("stored_image_filepaths" in df.columns)

if HAS_IMAGES:
    if "MANIFEST_PARQUET" not in globals() or (not isinstance(MANIFEST_PARQUET, str)) or (not os.path.exists(MANIFEST_PARQUET)):
        raise FileNotFoundError("MANIFEST_PARQUET missing/not found. Set MANIFEST_PARQUET to your image embeddings parquet.")

    img_manifest = pd.read_parquet(MANIFEST_PARQUET)
    # normalize
    if "image_path" not in img_manifest.columns or "vec_path" not in img_manifest.columns:
        raise ValueError("MANIFEST_PARQUET must contain image_path and vec_path.")

    # map: normalized image_path -> vec_path
    def normp(p: str) -> str:
        return os.path.normpath(str(p)).replace("\\", "/")

    path_to_vec = {}
    for _, r in img_manifest.iterrows():
        ip = r.get("image_path", None)
        vp = r.get("vec_path", None)
        if isinstance(ip, str) and isinstance(vp, str):
            path_to_vec[normp(ip)] = vp

    # Build prompt vectors using topics.csv keywords
    topics_csv = os.path.join(OUT_FINAL, "topics.csv")
    if not os.path.exists(topics_csv):
        raise FileNotFoundError(f"Missing topics.csv: {topics_csv}")

    topics_df = pd.read_csv(topics_csv)
    if "topic_id" not in topics_df.columns or "keywords" not in topics_df.columns:
        raise ValueError("topics.csv must contain topic_id and keywords columns.")

    # keep only topics used by this model (topic IDs in col order)
    want_topic_ids = sorted(set(topic_ids_in_col_order))
    topics_df = topics_df.copy()
    topics_df["topic_id"] = topics_df["topic_id"].astype(int)
    topics_df = topics_df[topics_df["topic_id"].isin(want_topic_ids)].sort_values("topic_id")

    topic_list = topics_df["topic_id"].tolist()
    keywords_list = topics_df["keywords"].fillna("").astype(str).tolist()

    import torch
    from sentence_transformers import SentenceTransformer

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    clip_text = SentenceTransformer("clip-ViT-B-32", device=DEVICE)

    prompts = [f"news photo of {kw}" for kw in keywords_list]
    P = clip_text.encode(prompts, show_progress_bar=True, convert_to_numpy=True).astype(np.float32)
    P = P / (np.linalg.norm(P, axis=1, keepdims=True) + 1e-12)  # (K, D)

    def softmax_stable(x, temp=0.07):
        x = (x / max(temp, 1e-6)).astype(np.float32)
        x = x - np.max(x)
        ex = np.exp(x)
        return ex / (np.sum(ex) + 1e-12)

    def split_paths(cell: str):
        s = str(cell) if cell is not None else ""
        s = s.strip()
        if not s or s.lower() == "nan":
            return []
        # support both "||" and "|" (pipe)
        if "||" in s:
            parts = [p.strip() for p in s.split("||")]
        elif "|" in s:
            parts = [p.strip() for p in s.split("|")]
        else:
            parts = [s.strip()]
        return [p for p in parts if p]

    img_topic_ids_col = []
    img_labels_col = []
    img_groups_col = []
    img_probs_col = []

    for i in tqdm(range(len(df)), desc="Curated image cols"):
        paths = split_paths(df.loc[i, "stored_image_filepaths"])
        if not paths:
            img_topic_ids_col.append("")
            img_labels_col.append("")
            img_groups_col.append("")
            img_probs_col.append("")
            continue

        out_ids = []
        out_lbls = []
        out_grps = []
        out_prs = []

        for p in paths:
            vp = path_to_vec.get(normp(p), None)
            if (vp is None) or (not isinstance(vp, str)) or (not os.path.exists(vp)):
                # keep alignment (blank placeholders for missing vec)
                out_ids.append("")
                out_lbls.append("")
                out_grps.append("")
                out_prs.append("")
                continue

            v = np.load(vp).astype(np.float32)
            v = v / (np.linalg.norm(v) + 1e-12)

            sims = (P @ v).astype(np.float32)         # (K,)
            pr = softmax_stable(sims, temp=IMAGE_TEMP)

            j = int(np.argmax(pr))
            top_tid = int(topic_list[j])
            top_prob = float(pr[j])

            out_ids.append(str(top_tid))
            out_lbls.append(label_for_tid(top_tid))
            out_grps.append(group_for_tid(top_tid))
            out_prs.append(PROB_FMT.format(top_prob))

        # join with "||" (your standard delimiter)
        img_topic_ids_col.append("||".join(out_ids))
        img_labels_col.append("||".join(out_lbls))
        img_groups_col.append("||".join(out_grps))
        img_probs_col.append("||".join(out_prs))

    df["curated_image_topic_ids"] = img_topic_ids_col
    df["curated_image_topic_labels"] = img_labels_col
    df["curated_image_group_names"] = img_groups_col
    df["curated_image_topic_probabilities"] = img_probs_col

else:
    print("No stored_image_filepaths column found → skipping curated image columns.")

# ---- save final ----
df.to_csv(FINAL_CURATED_CSV, index=False, encoding="utf-8")
print("Saved final curated CSV:", FINAL_CURATED_CSV)


## Export Long-Format CSVs (Text + Image)

**What this cell creates:**
1) **Text long format**: 1 row per (speech × topic) with probability
   → if N speeches and K topics ⇒ **N×K rows**
2) **Image long format**: 1 row per (image × topic) with probability
   → if M images and K topics ⇒ **M×K rows**

**Outputs:**
- `doc_topic_probs_long.csv` (can be huge)
- `image_topic_probs_long.csv` (can be huge)

**Notes:**
- For text: uses `probs` from BERTopic transform.
- For images: uses CLIP prompt vectors + cached image vectors from `MANIFEST_PARQUET`.
- If you want smaller output, set `KEEP_TOPN = 10` to store only top-10 topics per doc/image.


In [ ]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

assert "OUT_FINAL" in globals(), "OUT_FINAL not found. Run Cell 10."
assert "CSV_PATH" in globals(), "CSV_PATH not found."
assert "TEXT_COL" in globals(), "TEXT_COL not found."
assert "EMB_PATH" in globals(), "EMB_PATH not found."

KEEP_TOPN = None  # None = export ALL topics (N*K rows). Set e.g. 10 to export only top-10 per doc/image.

TEXT_LONG_CSV  = os.path.join(OUT_FINAL, "doc_topic_probs_long.csv")
IMAGE_LONG_CSV = os.path.join(OUT_FINAL, "image_topic_probs_long.csv")

# ---- load df + embeddings ----
df = pd.read_csv(CSV_PATH, encoding="utf-8", dtype=str, low_memory=False)

ID_COL = "id" if "id" in df.columns else ("ID" if "ID" in df.columns else None)
if ID_COL is None:
    raise ValueError("Input CSV must contain 'id' or 'ID' column.")
df[ID_COL] = df[ID_COL].astype(str)

docs = df[TEXT_COL].fillna("").astype(str).tolist()

emb = np.load(EMB_PATH)["X"].astype(np.float32)
if len(docs) != emb.shape[0]:
    raise ValueError(f"Docs count ({len(docs)}) != embedding rows ({emb.shape[0]}).")

# load final model
from bertopic import BERTopic
MODEL_FINAL_PATH = os.path.join(OUT_FINAL, "topic_model_text_only")
topic_model = BERTopic.load(MODEL_FINAL_PATH)

# transform (reuse if already in memory)
reuse_ok = ("topics" in globals()) and ("probs" in globals()) and (len(globals()["topics"]) == len(docs))
if reuse_ok:
    topics_arr = np.array(globals()["topics"], dtype=int)
    probs_arr  = np.array(globals()["probs"], dtype=np.float32)
else:
    topics_arr, probs_arr = topic_model.transform(docs, embeddings=emb)
    topics_arr = np.array(topics_arr, dtype=int)
    probs_arr  = np.array(probs_arr, dtype=np.float32)

# mapping prob columns -> topic_id
def build_col_to_topic_mapping(topics_arr, probs_arr):
    unique_tids = sorted({int(t) for t in set(topics_arr) if int(t) != -1})
    n_cols = probs_arr.shape[1]
    col_map = {}
    used_cols = set()
    for tid in unique_tids:
        idx = np.where(topics_arr == tid)[0]
        if len(idx) == 0:
            continue
        meanp = probs_arr[idx].mean(axis=0)
        order = np.argsort(-meanp)
        for c in order:
            if int(c) not in used_cols:
                col_map[int(c)] = int(tid)
                used_cols.add(int(c))
                break
    col_to_tid = [col_map.get(c, None) for c in range(n_cols)]
    valid_cols = [i for i, tid in enumerate(col_to_tid) if tid is not None]
    topic_ids_in_col_order = [col_to_tid[i] for i in valid_cols]
    return col_to_tid, valid_cols, topic_ids_in_col_order

col_to_tid, valid_cols, topic_ids_in_col_order = build_col_to_topic_mapping(topics_arr, probs_arr)
K = len(valid_cols)

rows = []
for i in tqdm(range(len(df)), desc="Text long export"):
    sid = df.loc[i, ID_COL]
    rowp = probs_arr[i, :][valid_cols]  # (K,)
    if KEEP_TOPN is None:
        idxs = range(K)
    else:
        idxs = np.argsort(-rowp)[:min(int(KEEP_TOPN), K)]
    for j in idxs:
        rows.append((sid, int(topic_ids_in_col_order[j]), float(rowp[j])))

text_long = pd.DataFrame(rows, columns=["id", "topic_id", "topic_prob"])
text_long.to_csv(TEXT_LONG_CSV, index=False, encoding="utf-8")
print("Saved:", TEXT_LONG_CSV, "| rows =", len(text_long))

if "stored_image_filepaths" in df.columns:
    if "MANIFEST_PARQUET" not in globals() or (not isinstance(MANIFEST_PARQUET, str)) or (not os.path.exists(MANIFEST_PARQUET)):
        raise FileNotFoundError("MANIFEST_PARQUET missing/not found. Set MANIFEST_PARQUET to your image embeddings parquet.")

    img_manifest = pd.read_parquet(MANIFEST_PARQUET)
    if "image_path" not in img_manifest.columns or "vec_path" not in img_manifest.columns:
        raise ValueError("MANIFEST_PARQUET must contain image_path and vec_path columns.")

    def normp(p: str) -> str:
        return os.path.normpath(str(p)).replace("\\", "/")

    path_to_vec = {}
    for _, r in img_manifest.iterrows():
        ip = r.get("image_path", None)
        vp = r.get("vec_path", None)
        if isinstance(ip, str) and isinstance(vp, str):
            path_to_vec[normp(ip)] = vp

    topics_csv = os.path.join(OUT_FINAL, "topics.csv")
    if not os.path.exists(topics_csv):
        raise FileNotFoundError(f"Missing topics.csv: {topics_csv}")
    topics_df = pd.read_csv(topics_csv)

    # keywords for prompt vectors
    topics_df["topic_id"] = topics_df["topic_id"].astype(int)
    want_tids = sorted(set(topic_ids_in_col_order))
    topics_df = topics_df[topics_df["topic_id"].isin(want_tids)].sort_values("topic_id")
    topic_list = topics_df["topic_id"].tolist()
    keywords_list = topics_df["keywords"].fillna("").astype(str).tolist()

    import torch
    from sentence_transformers import SentenceTransformer

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    clip_text = SentenceTransformer("clip-ViT-B-32", device=DEVICE)

    prompts = [f"news photo of {kw}" for kw in keywords_list]
    P = clip_text.encode(prompts, show_progress_bar=True, convert_to_numpy=True).astype(np.float32)
    P = P / (np.linalg.norm(P, axis=1, keepdims=True) + 1e-12)  # (K, D)

    IMAGE_TEMP = 0.07
    def softmax_stable(x, temp=0.07):
        x = (x / max(temp, 1e-6)).astype(np.float32)
        x = x - np.max(x)
        ex = np.exp(x)
        return ex / (np.sum(ex) + 1e-12)

    def split_paths(cell: str):
        s = str(cell) if cell is not None else ""
        s = s.strip()
        if not s or s.lower() == "nan":
            return []
        if "||" in s:
            parts = [p.strip() for p in s.split("||")]
        elif "|" in s:
            parts = [p.strip() for p in s.split("|")]
        else:
            parts = [s.strip()]
        return [p for p in parts if p]

    img_rows = []
    for i in tqdm(range(len(df)), desc="Image long export"):
        sid = df.loc[i, ID_COL]
        paths = split_paths(df.loc[i, "stored_image_filepaths"])
        for p in paths:
            vp = path_to_vec.get(normp(p), None)
            if (vp is None) or (not isinstance(vp, str)) or (not os.path.exists(vp)):
                continue

            v = np.load(vp).astype(np.float32)
            v = v / (np.linalg.norm(v) + 1e-12)

            sims = (P @ v).astype(np.float32)  # (K,)
            pr = softmax_stable(sims, temp=IMAGE_TEMP)

            if KEEP_TOPN is None:
                idxs = range(len(topic_list))
            else:
                idxs = np.argsort(-pr)[:min(int(KEEP_TOPN), len(topic_list))]

            for j in idxs:
                img_rows.append((sid, p, int(topic_list[j]), float(pr[j])))

    image_long = pd.DataFrame(img_rows, columns=["id", "image_path", "topic_id", "topic_prob"])
    image_long.to_csv(IMAGE_LONG_CSV, index=False, encoding="utf-8")
    print("Saved:", IMAGE_LONG_CSV, "| rows =", len(image_long))

else:
    print("No stored_image_filepaths column found → skipping image long export.")


## Build Curated Columns (Final Wide CSV)

- Create one final CSV (same rows as input) with these curated fields filled:

### Text curated fields
- `curated_topic_id`
- `curated_text_topic_label`
- `curated_text_topic_group`
- `curated_topic_probabilities`  *(stores all topic probs for that row)*

### Image curated fields
- `curated_image_topic_ids`
- `curated_image_topic_labels`
- `curated_image_group_names`
- `curated_image_topic_probabilities`

1) **Text:** for each speech row, take the BERTopic probability vector `probs`
   - `curated_topic_id` = argmax topic
   - label/group come from the correct dictionary (MID/Kremlin + EN/RU)
   - `curated_topic_probabilities` = a list []

2) **Images:** use your **image-long** output (or recompute quickly if needed)
   - for each image, argmax topic from its probability vector
   - then aggregate per speech as pipe-separated lists in the same order as `stored_image_filepaths`

**Output:**
- `final_with_curated_columns.csv`

**Requires you already set (in earlier config cell):**
- `DATASET_KEY` ∈ {"mid_en","mid_ru","kremlin_en","kremlin_ru"}
- Topic dictionaries exist: `MID_EN_TOPIC_DICT`, `MID_RU_TOPIC_DICT`, `KREMLIN_EN_TOPIC_DICT`, `KREMLIN_RU_TOPIC_DICT`


In [ ]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

assert "OUT_FINAL" in globals(), "OUT_FINAL not found. Run prior cells."
assert "CSV_PATH" in globals(), "CSV_PATH not found."
assert "TEXT_COL" in globals(), "TEXT_COL not found."
assert "DATASET_KEY" in globals(), "DATASET_KEY not found. Set it in config."
assert "EMB_PATH" in globals(), "EMB_PATH not found."

FINAL_WIDE_CSV = os.path.join(OUT_FINAL, "final_with_curated_columns.csv")

df = pd.read_csv(CSV_PATH, encoding="utf-8", dtype=str, low_memory=False)

ID_COL = "id" if "id" in df.columns else ("ID" if "ID" in df.columns else None)
if ID_COL is None:
    raise ValueError("Input CSV must contain 'id' or 'ID' column.")
df[ID_COL] = df[ID_COL].astype(str)

docs = df[TEXT_COL].fillna("").astype(str).tolist()
emb  = np.load(EMB_PATH)["X"].astype(np.float32)

# load model + transform (reuse if already computed)
from bertopic import BERTopic
MODEL_FINAL_PATH = os.path.join(OUT_FINAL, "topic_model_text_only")
topic_model = BERTopic.load(MODEL_FINAL_PATH)

reuse_ok = ("topics" in globals()) and ("probs" in globals()) and (len(globals()["topics"]) == len(docs))
if reuse_ok:
    topics_arr = np.array(globals()["topics"], dtype=int)
    probs_arr  = np.array(globals()["probs"], dtype=np.float32)
else:
    topics_arr, probs_arr = topic_model.transform(docs, embeddings=emb)
    topics_arr = np.array(topics_arr, dtype=int)
    probs_arr  = np.array(probs_arr, dtype=np.float32)

# mapping prob columns -> topic_id
def build_col_to_topic_mapping(topics_arr, probs_arr):
    unique_tids = sorted({int(t) for t in set(topics_arr) if int(t) != -1})
    n_cols = probs_arr.shape[1]
    col_map = {}
    used_cols = set()
    for tid in unique_tids:
        idx = np.where(topics_arr == tid)[0]
        if len(idx) == 0:
            continue
        meanp = probs_arr[idx].mean(axis=0)
        order = np.argsort(-meanp)
        for c in order:
            if int(c) not in used_cols:
                col_map[int(c)] = int(tid)
                used_cols.add(int(c))
                break
    col_to_tid = [col_map.get(c, None) for c in range(n_cols)]
    valid_cols = [i for i, tid in enumerate(col_to_tid) if tid is not None]
    topic_ids_in_col_order = [col_to_tid[i] for i in valid_cols]
    return valid_cols, topic_ids_in_col_order

valid_cols, topic_ids_in_col_order = build_col_to_topic_mapping(topics_arr, probs_arr)
K = len(valid_cols)

# choose correct topic dict for THIS dataset
TOPIC_DICT_MAP = {
    "mid_en":      globals().get("MID_EN_TOPIC_DICT", None),
    "mid_ru":      globals().get("MID_RU_TOPIC_DICT", None),
    "kremlin_en":  globals().get("KREMLIN_EN_TOPIC_DICT", None),
    "kremlin_ru":  globals().get("KREMLIN_RU_TOPIC_DICT", None),
}
TOPIC_DICT = TOPIC_DICT_MAP.get(DATASET_KEY)
if TOPIC_DICT is None:
    raise ValueError(f"Topic dict missing for DATASET_KEY={DATASET_KEY}. Define the correct *_TOPIC_DICT first.")

def label_group(tid: int):
    d = TOPIC_DICT.get(int(tid), {})
    return (d.get("label", ""), d.get("group", ""))

cur_topic_id = []
cur_label    = []
cur_group    = []
cur_probs    = []

for i in tqdm(range(len(df)), desc="Curated text columns"):
    rowp = probs_arr[i, :][valid_cols]  # length K
    j = int(np.argmax(rowp))
    top_tid = int(topic_ids_in_col_order[j])
    lbl, grp = label_group(top_tid)

    # store full distribution as topic:prob|topic:prob...
    parts = [f"{int(tid)}:{float(p):.6f}" for tid, p in zip(topic_ids_in_col_order, rowp)]
    cur_topic_id.append(top_tid)
    cur_label.append(lbl)
    cur_group.append(grp)
    cur_probs.append("|".join(parts))

df["curated_topic_id"] = [str(x) for x in cur_topic_id]
df["curated_text_topic_label"] = cur_label
df["curated_text_topic_group"] = cur_group
df["curated_topic_probabilities"] = cur_probs

if "stored_image_filepaths" in df.columns:
    # prefer using long CSV if it exists
    IMAGE_LONG_CSV = os.path.join(OUT_FINAL, "image_topic_probs_long.csv")
    if os.path.exists(IMAGE_LONG_CSV):
        img_long = pd.read_csv(IMAGE_LONG_CSV, dtype={"id": str, "image_path": str, "topic_id": int, "topic_prob": float})
    else:
        print("image_topic_probs_long.csv not found. Skipping image curated columns.")
        img_long = None

    def split_paths(cell: str):
        s = str(cell) if cell is not None else ""
        s = s.strip()
        if not s or s.lower() == "nan":
            return []
        if "||" in s:
            parts = [p.strip() for p in s.split("||")]
        elif "|" in s:
            parts = [p.strip() for p in s.split("|")]
        else:
            parts = [s.strip()]
        return [p for p in parts if p]

    if img_long is not None and not img_long.empty:
        # For each (id, image_path), choose top topic by max prob
        img_long_sorted = img_long.sort_values(["id", "image_path", "topic_prob"], ascending=[True, True, False])
        best = img_long_sorted.groupby(["id", "image_path"], as_index=False).first()

        # Build quick lookup: (id -> dict(image_path -> (tid,prob)))
        img_map = {}
        for _, r in best.iterrows():
            sid = str(r["id"])
            ip  = str(r["image_path"])
            tid = int(r["topic_id"])
            pr  = float(r["topic_prob"])
            img_map.setdefault(sid, {})[ip] = (tid, pr)

        img_topic_ids = []
        img_labels    = []
        img_groups    = []
        img_probs     = []

        for i in tqdm(range(len(df)), desc="Curated image columns"):
            sid = str(df.loc[i, ID_COL])
            paths = split_paths(df.loc[i, "stored_image_filepaths"])
            ids_list, lbl_list, grp_list, pr_list = [], [], [], []

            for p in paths:
                tid_pr = img_map.get(sid, {}).get(p, None)
                if tid_pr is None:
                    ids_list.append("")
                    lbl_list.append("")
                    grp_list.append("")
                    pr_list.append("")
                else:
                    tid, pr = tid_pr
                    lbl, grp = label_group(tid)  # same topic dict
                    ids_list.append(str(tid))
                    lbl_list.append(lbl)
                    grp_list.append(grp)
                    pr_list.append(f"{pr:.6f}")

            img_topic_ids.append("||".join(ids_list))
            img_labels.append("||".join(lbl_list))
            img_groups.append("||".join(grp_list))
            img_probs.append("||".join(pr_list))

        df["curated_image_topic_ids"] = img_topic_ids
        df["curated_image_topic_labels"] = img_labels
        df["curated_image_group_names"] = img_groups
        df["curated_image_topic_probabilities"] = img_probs

# ---- save final ----
df.to_csv(FINAL_WIDE_CSV, index=False, encoding="utf-8")
print("Saved final wide CSV with curated columns →", FINAL_WIDE_CSV)
